In [ ]:
"""
Imports and global setup.
Pure Python + OpenCV + NumPy + SciPy, no CUDA or ROS dependency.
"""
import numpy as np
import cv2
import matplotlib; matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import scipy
from scipy.spatial import KDTree, ConvexHull
from scipy.optimize import minimize
from scipy.interpolate import griddata
from scipy.ndimage import uniform_filter, gaussian_filter1d
from scipy.stats import gaussian_kde
from sklearn.cluster import DBSCAN
import imageio.v2 as imageio
import os, sys, time, json, warnings, copy
from pathlib import Path
from collections import deque, defaultdict
import uuid as _uuid

warnings.filterwarnings('ignore')
np.random.seed(42)

# Publication-quality matplotlib style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.labelsize': 10,
    'axes.titlesize': 10,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 1.2,
})

OUTPUT_DIR = Path('uav_simulation_output')
DIRS = {k: OUTPUT_DIR / v for k, v in {
    'root': '.', 'videos': 'videos', 'trajectories': 'trajectories',
    'metrics': 'metrics', 'features': 'features',
    'keyframes': 'keyframes',
    'evaluation': 'evaluation', 'slam_maps': 'slam_maps'}.items()}
DIRS['root'] = OUTPUT_DIR
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

try:
    import open3d as o3d; HAS_OPEN3D = True; print("open3d available")
except ImportError:
    HAS_OPEN3D = False; print("open3d not found, using built-in PLY reader")

print(f"NumPy {np.__version__}  OpenCV {cv2.__version__}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


In [ ]:
"""
Point cloud loading, terrain model, and trunk-zone upsampling.

Trunk-zone upsampling (0.8-2.0 m above terrain, factor=4, sigma_xy=2 cm)
increases point density in the stem zone from roughly 5 pts/m^2 to roughly
20 pts/m^2. RGB colour jitter on the upsampled points (sigma=0.04) introduces
bark texture variation that SIFT can pick up as distinct keypoints.
The original cloud is kept in pcp_original for evaluation against the
reconstructed result later on.
"""

# Path to the input PLY point cloud.
POINT_CLOUD_PATH = 'final_combined_scene.ply'

def read_ply_numpy(filepath):
    with open(filepath, 'rb') as f:
        header = []
        while True:
            line = f.readline().decode('utf-8', errors='replace').strip()
            header.append(line)
            if line == 'end_header': break
        ht = '\n'.join(header)
        binary = 'binary_little_endian' in ht or 'binary_big_endian' in ht
        big_endian = 'binary_big_endian' in ht
        n_verts = 0
        for h in header:
            if h.startswith('element vertex'): n_verts = int(h.split()[-1]); break
        props, ptypes = [], []
        in_v = False
        for h in header:
            if h.startswith('element vertex'): in_v = True; continue
            if h.startswith('element') and 'vertex' not in h: in_v = False; continue
            if in_v and h.startswith('property'):
                p = h.split(); ptypes.append(p[1]); props.append(p[2])
        tmap = {'float': np.float32, 'float32': np.float32, 'float64': np.float64,
                'double': np.float64, 'uchar': np.uint8, 'uint8': np.uint8,
                'int': np.int32, 'int32': np.int32, 'short': np.int16,
                'ushort': np.uint16, 'uint': np.uint32}
        bmap = {'float': 4, 'float32': 4, 'float64': 8, 'double': 8,
                'uchar': 1, 'uint8': 1, 'int': 4, 'int32': 4,
                'short': 2, 'ushort': 2, 'uint': 4}
        if binary:
            rb = sum(bmap.get(t, 4) for t in ptypes)
            raw = np.frombuffer(f.read(n_verts * rb), dtype=np.uint8)
            dt = np.dtype([(p, tmap.get(t, np.float32)) for p, t in zip(props, ptypes)])
            data = raw.view(dt)
            if big_endian: data = data.byteswap().newbyteorder()
        else:
            rows = [f.readline().decode().split() for _ in range(n_verts)]
            arr = np.array(rows, dtype=np.float32)
            data = {p: arr[:, i] for i, p in enumerate(props)}

    def _get(data, keys):
        for k in keys:
            try:
                if isinstance(data, np.ndarray):
                    if k in data.dtype.names: return data[k].astype(np.float64)
                else:
                    if k in data: return np.array(data[k], dtype=np.float64)
            except: pass
        return None

    pts = np.stack([_get(data, ['x']), _get(data, ['y']), _get(data, ['z'])], axis=1)
    r = _get(data, ['red', 'r']); g = _get(data, ['green', 'g']); b = _get(data, ['blue', 'b'])
    cols = np.stack([r / 255, g / 255, b / 255], axis=1) if r is not None else np.full((len(pts), 3), 0.5)
    return pts.astype(np.float64), cols.astype(np.float64)


def remove_outliers(pts, cols, k=20, sigma=2.0):
    tree = KDTree(pts)
    d, _ = tree.query(pts, k=k + 1)
    m = d[:, 1:].mean(axis=1)
    mask = m < m.mean() + sigma * m.std()
    return pts[mask], cols[mask]


class TerrainModel:
    def __init__(self, pts, cell=0.5):
        self.cell = cell
        xm, ym = pts[:, 0].min(), pts[:, 1].min()
        xM, yM = pts[:, 0].max(), pts[:, 1].max()
        nx = int((xM - xm) / cell) + 2; ny = int((yM - ym) / cell) + 2
        self.xs = xm + np.arange(nx) * cell
        self.ys = ym + np.arange(ny) * cell
        self.xm, self.ym = xm, ym
        Z = np.full((ny, nx), np.inf)
        xi = np.clip(((pts[:, 0] - xm) / cell).astype(int), 0, nx - 1)
        yi = np.clip(((pts[:, 1] - ym) / cell).astype(int), 0, ny - 1)
        for k in range(len(pts)):
            if pts[k, 2] < Z[yi[k], xi[k]]: Z[yi[k], xi[k]] = pts[k, 2]
        v = np.isfinite(Z)
        if not v.all():
            gy, gx = np.mgrid[0:ny, 0:nx]
            Z[~v] = griddata((gy[v], gx[v]), Z[v], (gy[~v], gx[~v]), method='nearest')
        G = Z.copy()
        for win in [3, 5, 7, 11, 15]:
            S = uniform_filter(G, size=win)
            G = np.where(G - S > 0.3 * win * cell, S, G)
        self.Z = G
        z_t = self.query(pts[:, :2])
        self.ground_mask = (pts[:, 2] - z_t) < 0.35
        print(f"DTM grid: {nx}x{ny}, ground points: {self.ground_mask.sum():,}")

    def query(self, xy):
        scalar = (xy.ndim == 1)
        if scalar: xy = xy[None]
        ny, nx = self.Z.shape
        xi = np.clip((xy[:, 0] - self.xm) / self.cell, 0, nx - 1.001)
        yi = np.clip((xy[:, 1] - self.ym) / self.cell, 0, ny - 1.001)
        x0 = xi.astype(int); x1 = np.minimum(x0 + 1, nx - 1)
        y0 = yi.astype(int); y1 = np.minimum(y0 + 1, ny - 1)
        fx, fy = xi - x0, yi - y0
        z = (self.Z[y0, x0] * (1 - fx) * (1 - fy) + self.Z[y0, x1] * fx * (1 - fy) +
             self.Z[y1, x0] * (1 - fx) * fy + self.Z[y1, x1] * fx * fy)
        return float(z[0]) if scalar else z


class PointCloudProcessor:
    def __init__(self, path):
        print(f"Loading point cloud: {path}")
        if not os.path.exists(path):
            raise FileNotFoundError(f"Not found: {path}\nUpdate POINT_CLOUD_PATH.")
        t0 = time.time()
        if HAS_OPEN3D:
            pcd = o3d.io.read_point_cloud(path)
            pts = np.asarray(pcd.points)
            cols = np.asarray(pcd.colors) if pcd.has_colors() else np.full((len(pts), 3), 0.5)
        else:
            pts, cols = read_ply_numpy(path)
        print(f"Raw points: {len(pts):,} ({time.time()-t0:.1f}s)")
        pts, cols = remove_outliers(pts, cols)
        print(f"After outlier removal: {len(pts):,} points")
        self.pts = pts; self.cols = cols
        self.bounds = {'min': pts.min(0), 'max': pts.max(0),
                       'center': pts.mean(0), 'extent': pts.max(0) - pts.min(0)}
        self.kdtree = KDTree(pts)
        self.terrain = TerrainModel(pts)
        self._save_overview()
        ext = self.bounds['extent']
        print(f"Loaded {len(pts):,} points, extent {ext[0]:.1f} x {ext[1]:.1f} x {ext[2]:.1f} m")
        print(f"Ground-plane density: {len(pts)/(ext[0]*ext[1]):.0f} pts/m^2")

    def _save_overview(self):
        n = min(60000, len(self.pts))
        idx = np.random.choice(len(self.pts), n, replace=False)
        p, c = self.pts[idx], np.clip(self.cols[idx], 0, 1)
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        for ax, (a, b), xl, yl, ti in zip(axes, [(0, 1), (0, 2), (1, 2)],
                                            ['X', 'X', 'Y'], ['Y', 'Z', 'Z'],
                                            ['Top (XY)', 'Side (XZ)', 'Front (YZ)']):
            ax.scatter(p[:, a], p[:, b], c=c, s=0.3, alpha=0.5)
            ax.set_xlabel(f'{xl} (m)'); ax.set_ylabel(f'{yl} (m)')
            ax.set_title(ti); ax.set_aspect('equal')
        fig.suptitle('Point cloud overview - Zagros forest plot', fontsize=14, fontweight='bold')
        plt.tight_layout()
        out = DIRS['evaluation'] / '01_point_cloud_overview.png'
        fig.savefig(out, dpi=150); plt.show(); plt.close(fig)
        print(f"Saved {out}")


pcp = PointCloudProcessor(POINT_CLOUD_PATH)
pcp_original = copy.copy(pcp)  # unmodified copy, kept for evaluation against the reconstruction

print("Upsampling trunk zone (factor=4, sigma_xy=2cm, RGB jitter)...")

def upsample_trunk_zone(pts, cols, terrain, factor=4,
                         sigma_xy=0.02, rgb_jitter=0.04,
                         z_lo=0.8, z_hi=2.0):
    """
    For every point within [z_lo, z_hi] above terrain, add `factor` extra
    neighbours: Gaussian XY offset (sigma_xy), Z unchanged, colour jitter
    (rgb_jitter). Raises point density in the stem zone several-fold.
    """
    hab = pts[:, 2] - terrain.query(pts[:, :2])
    mask = (hab >= z_lo) & (hab <= z_hi)
    idx_m = np.where(mask)[0]
    if len(idx_m) == 0:
        print("Warning: no trunk-zone points found")
        return pts, cols
    extra_pts, extra_cols = [], []
    for i in idx_m:
        for _ in range(factor):
            off = np.random.normal(0, sigma_xy, 2)
            extra_pts.append([pts[i, 0] + off[0], pts[i, 1] + off[1], pts[i, 2]])
            extra_cols.append(np.clip(cols[i] + np.random.normal(0, rgb_jitter, 3), 0, 1))
    new_pts = np.vstack([pts, np.array(extra_pts)])
    new_cols = np.vstack([cols, np.array(extra_cols)])
    return new_pts, new_cols

pcp.pts, pcp.cols = upsample_trunk_zone(pcp.pts, pcp.cols, pcp.terrain)
pcp.kdtree = KDTree(pcp.pts)
added = len(pcp.pts) - len(pcp_original.pts)
print(f"{len(pcp_original.pts):,} -> {len(pcp.pts):,} points (+{added:,} trunk-zone points)")


In [ ]:
"""
UAV platform selection.
"""
PLATFORMS = {
    '1': {'id': 'mavic_pro', 'name': 'DJI Mavic Pro',
          'cam': {'W': 1920, 'H': 1080, 'fps': 30, 'fx': 1230., 'fy': 1230., 'cx': 960., 'cy': 540., 'fov_h': 78.8},
          'op_speed': 2.0, 'weight': 0.743,
          'sensors': {'a_sig': 0.002, 'g_sig': 0.0001, 'baro_sig': 0.10,
                      'baro_drift': 0.003, 'sonar_range': (0.3, 5.0), 'sonar_sig': 0.02}},
    '2': {'id': 'parrot_anafi', 'name': 'Parrot Anafi',
          'cam': {'W': 3840, 'H': 2160, 'fps': 30, 'fx': 2800., 'fy': 2800., 'cx': 1920., 'cy': 1080., 'fov_h': 69.},
          'op_speed': 1.5, 'weight': 0.320,
          'sensors': {'a_sig': 0.003, 'g_sig': 0.0002, 'baro_sig': 0.12,
                      'baro_drift': 0.004, 'sonar_range': (0.5, 5.0), 'sonar_sig': 0.03}},
    '3': {'id': 'custom', 'name': 'Custom Lightweight',
          'cam': {'W': 1280, 'H': 720, 'fps': 60, 'fx': 740., 'fy': 740., 'cx': 640., 'cy': 360., 'fov_h': 90.},
          'op_speed': 1.0, 'weight': 0.200,
          'sensors': {'a_sig': 0.005, 'g_sig': 0.0003, 'baro_sig': 0.15,
                      'baro_drift': 0.006, 'sonar_range': (0.3, 4.0), 'sonar_sig': 0.04}},
}

print("Select UAV platform:")
for k, v in PLATFORMS.items():
    c = v['cam']
    print(f"  [{k}] {v['name']}")
    print(f"      {c['W']}x{c['H']} @ {c['fps']}fps  FOV={c['fov_h']} deg  speed={v['op_speed']}m/s")
print("  [4] Custom")

choice = input("Choice [1/2/3/4, default=1]: ").strip() or '1'
if choice == '4':
    W = int(input("  Width [px]: ").strip() or 1920)
    H = int(input("  Height [px]: ").strip() or 1080)
    fx = float(input("  fx [px]: ").strip() or 1230)
    sp = float(input("  Speed [m/s]: ").strip() or 2.0)
    cfg = {'id': 'custom_user', 'name': 'Custom',
           'cam': {'W': W, 'H': H, 'fps': 30, 'fx': fx, 'fy': fx, 'cx': W/2, 'cy': H/2, 'fov_h': 78.8},
           'op_speed': sp, 'weight': 0.5,
           'sensors': {'a_sig': 0.002, 'g_sig': 0.0001, 'baro_sig': 0.10,
                       'baro_drift': 0.003, 'sonar_range': (0.3, 5.0), 'sonar_sig': 0.02}}
else:
    cfg = PLATFORMS.get(choice, PLATFORMS['1'])

cam = cfg['cam']
K_mat = np.array([[cam['fx'], 0, cam['cx']], [0, cam['fy'], cam['cy']], [0, 0, 1]], dtype=np.float64)
cfg['K'] = K_mat; cfg['collision_radius'] = 0.65
uav = cfg

print(f"Selected: {cfg['name']}  {cam['W']}x{cam['H']}  fx={cam['fx']}")
print(f"K =\n{K_mat}")


In [ ]:
"""
Mission planning and Two-Phase Trajectory Anchoring (2PTA), phase 1.

2PTA phase 1 (pre-flight) checks every waypoint against the point cloud and
nudges colliding ones radially outward, with Gaussian smoothing across the
loop. Phase 2 (in-flight, see the autonomous flight cell) re-checks after
every SLAM loop closure. Parameters below are tuned for dense Zagros oak
coppice stands.
"""
from scipy.ndimage import gaussian_filter1d

print("Mission configuration")
ext = pcp.bounds['extent']; ctr = pcp.bounds['center']
max_r = min(ext[0], ext[1]) / 2 * 0.8
print(f"Cloud extent: {ext[0]:.1f} x {ext[1]:.1f} m")
print(f"Recommended radius range: 5 - {max_r:.0f} m")

R = float(np.clip(float(input(f"Circle radius [m, default=15]: ").strip() or 15), 3, max_r))
h_above = float(np.clip(float(input("Height above terrain [m, default=1.0]: ").strip() or 1.0), 0.3, 5.0))
n_wp = int(np.clip(int(input("Waypoints [default=120]: ").strip() or 120), 30, 500))
direction = (input("Direction [CCW/CW, default=CCW]: ").strip().upper() or 'CCW')
if direction not in ['CW', 'CCW']: direction = 'CCW'
speed = float(np.clip(float(input("Speed [m/s, default=2.0]: ").strip() or 2.0), 0.3, 5.0))

print(f"R={R}m  h={h_above}m  N={n_wp}  {direction}  speed={speed}m/s")

# Interactive start point selection
n = min(80000, len(pcp.pts))
idx = np.random.choice(len(pcp.pts), n, replace=False)
p_disp = pcp.pts[idx]; c_disp = np.clip(pcp.cols[idx], 0, 1)
t_z = pcp.terrain.query(p_disp[:, :2])
ht = p_disp[:, 2] - t_z

fig_sel, ax_sel = plt.subplots(figsize=(13, 10))
sc = ax_sel.scatter(p_disp[:, 0], p_disp[:, 1], c=ht, cmap='RdYlGn',
                     s=0.3, alpha=0.5, vmin=0, vmax=15)
plt.colorbar(sc, ax=ax_sel, label='Height above terrain (m)', shrink=0.7)
ax_sel.add_patch(Circle(ctr[:2], R, fill=False, edgecolor='cyan', lw=2, ls='--'))
ax_sel.plot(*ctr[:2], 'c+', ms=15, mew=2)
ax_sel.set_xlabel('X (m)'); ax_sel.set_ylabel('Y (m)')
ax_sel.set_title('Click to set start point on circle perimeter\n(close window to confirm)',
                  fontsize=12, fontweight='bold')
ax_sel.set_aspect('equal'); fig_sel.tight_layout()

clicked_start = [None]; drawn = [None]

def _ctr(sxy):
    v = ctr[:2] - sxy; n = np.linalg.norm(v)
    return sxy + R * (v / n if n > 1e-6 else np.array([1., 0.]))

def on_click(ev):
    if ev.inaxes != ax_sel or ev.button != 1: return
    sxy = np.array([ev.xdata, ev.ydata]); cxy = _ctr(sxy)
    clicked_start[0] = sxy
    if drawn[0]:
        for a in drawn[0]: a.remove()
    c1, = ax_sel.plot(*sxy, 'ro', ms=12, zorder=10)
    c2, = ax_sel.plot(*cxy, 'r+', ms=16, mew=3, zorder=10)
    c3 = Circle(cxy, R, fill=False, edgecolor='red', lw=2.5); ax_sel.add_patch(c3)
    drawn[0] = [c1, c2, c3]
    ax_sel.set_title(f'Start:({sxy[0]:.1f},{sxy[1]:.1f})  Center:({cxy[0]:.1f},{cxy[1]:.1f})\nClose to confirm.', fontsize=11, fontweight='bold')
    fig_sel.canvas.draw()

fig_sel.canvas.mpl_connect('button_press_event', on_click)
plt.show(block=True)

if clicked_start[0] is None:
    clicked_start[0] = ctr[:2] + np.array([R, 0.])
    print(f"No click registered, using default start: {clicked_start[0]}")

start_xy = clicked_start[0]; circle_center_xy = _ctr(start_xy)
start_angle = float(np.arctan2(start_xy[1] - circle_center_xy[1], start_xy[0] - circle_center_xy[0]))
print(f"Start: {start_xy}  Center: {circle_center_xy}")

# Generate waypoints around the circle
angles = np.linspace(0, 2 * np.pi, n_wp, endpoint=False)
if direction == 'CW': angles = -angles
angles = start_angle + angles
waypoints = np.zeros((n_wp, 3))
for i, th in enumerate(angles):
    x = circle_center_xy[0] + R * np.cos(th)
    y = circle_center_xy[1] + R * np.sin(th)
    waypoints[i] = [x, y, pcp.terrain.query(np.array([x, y])) + h_above]
circle_center_3d = np.array([*circle_center_xy, waypoints[0, 2]])

# 2PTA phase 1: pre-flight waypoint deformation
print("Running 2PTA phase 1 (pre-flight waypoint deformation)...")
WD_SAFETY = 1.0; WD_INFLUENCE = 2.5; WD_HEIGHT_BND = 0.8
WD_DEFORM_MAX = 4.0; WD_SMOOTH_SIG = 4; COL_RADIUS = 0.65

waypoints_nominal = waypoints.copy(); waypoints_deformed = waypoints.copy()
deform_count = 0

for wi in range(len(waypoints)):
    wp = waypoints_deformed[wi]
    wp_z = pcp.terrain.query(wp[:2]) + h_above
    nearby = pcp.kdtree.query_ball_point(np.array([wp[0], wp[1], wp_z]), WD_INFLUENCE)
    max_intr = 0.; push_dir = np.zeros(2)
    for idx in nearby:
        pt = pcp.pts[idx]
        if abs(pt[2] - wp_z) > WD_HEIGHT_BND: continue
        d_xy = float(np.linalg.norm(pt[:2] - wp[:2])); needed = COL_RADIUS + WD_SAFETY
        if d_xy < needed:
            intr = needed - d_xy
            if intr > max_intr: max_intr = intr; push_dir = (wp[:2] - pt[:2]) / (d_xy + 1e-8)
    if max_intr > 0:
        rv = wp[:2] - circle_center_xy; rn = np.linalg.norm(rv) + 1e-8
        blend = 0.6 * (rv / rn) + 0.4 * push_dir
        blend /= (np.linalg.norm(blend) + 1e-8)
        shift = float(np.clip(max_intr * 2.0, 0, WD_DEFORM_MAX))
        new_xy = wp[:2] + blend * shift
        waypoints_deformed[wi] = np.array([new_xy[0], new_xy[1], pcp.terrain.query(new_xy) + h_above])
        deform_count += 1

wx = gaussian_filter1d(waypoints_deformed[:, 0], sigma=WD_SMOOTH_SIG, mode='wrap')
wy = gaussian_filter1d(waypoints_deformed[:, 1], sigma=WD_SMOOTH_SIG, mode='wrap')
for wi in range(len(waypoints_deformed)):
    waypoints_deformed[wi] = [wx[wi], wy[wi], pcp.terrain.query(np.array([wx[wi], wy[wi]])) + h_above]

waypoints = waypoints_deformed.copy()
shifts = np.linalg.norm(waypoints_deformed[:, :2] - waypoints_nominal[:, :2], axis=1)
print(f"Deformed {deform_count}/{len(waypoints)} waypoints, mean shift {shifts.mean():.3f}m, max {shifts.max():.3f}m")

# Flight path visualisation
fig2 = plt.figure(figsize=(16, 7))
ax3d = fig2.add_subplot(121, projection='3d')
n2 = min(20000, len(pcp.pts)); idx2 = np.random.choice(len(pcp.pts), n2, replace=False)
ax3d.scatter(pcp.pts[idx2, 0], pcp.pts[idx2, 1], pcp.pts[idx2, 2],
             c=np.clip(pcp.cols[idx2], 0, 1), s=0.1, alpha=0.15)
ax3d.plot(waypoints_nominal[:, 0], waypoints_nominal[:, 1], waypoints_nominal[:, 2], 'c--', lw=1, alpha=0.5, label='Nominal')
ax3d.plot(waypoints[:, 0], waypoints[:, 1], waypoints[:, 2], 'r-', lw=2, label='Deformed')
ax3d.plot(*waypoints[0], 'go', ms=10, label='Start'); ax3d.plot(*circle_center_3d, 'b+', ms=12)
ax3d.set_xlabel('X'); ax3d.set_ylabel('Y'); ax3d.set_zlabel('Z')
ax3d.set_title('3D path'); ax3d.legend(fontsize=8)

ax2d = fig2.add_subplot(122)
ax2d.scatter(p_disp[:, 0], p_disp[:, 1], c=c_disp, s=0.2, alpha=0.25)
ax2d.plot(waypoints_nominal[:, 0], waypoints_nominal[:, 1], 'c--', lw=1, alpha=0.5, label='Nominal')
ax2d.plot(waypoints[:, 0], waypoints[:, 1], 'r-', lw=2, label='Deformed')
ax2d.plot(*waypoints[0, :2], 'go', ms=10, label='Start')
ax2d.plot(*circle_center_xy, 'b+', ms=12, mew=2)
for i in [0, n_wp // 4, n_wp // 2, 3 * n_wp // 4]:
    j = (i + 3) % n_wp
    ax2d.annotate('', xy=(waypoints[j, 0], waypoints[j, 1]),
                  xytext=(waypoints[i, 0], waypoints[i, 1]),
                  arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
ax2d.set_xlabel('X (m)'); ax2d.set_ylabel('Y (m)'); ax2d.set_aspect('equal'); ax2d.legend()
ax2d.set_title(f'Top view, R={R}m, {direction}')
fig2.suptitle(f'Flight path: {n_wp} waypoints, h={h_above}m, deformed={deform_count}, speed={speed}m/s',
              fontsize=13, fontweight='bold')
plt.tight_layout()
fig2.savefig(DIRS['evaluation'] / '02_flight_path.png', dpi=150); plt.show(); plt.close(fig2)


In [ ]:
"""
Virtual camera: adaptive splat rendering, distance-based splat radius,
TELEA inpainting for hole-filling. Renders directly from the full-resolution
point cloud, no subsampling.
"""

class VirtualCamera:
    def __init__(self, K, W, H, pts, cols, kdtree, max_dist=22.0):
        self.K = K; self.W = W; self.H = H
        self.pts = pts; self.cols = cols; self.kd = kdtree
        self.max_dist = max_dist

    def _world_to_camera(self, yaw):
        cy, sy = np.cos(yaw), np.sin(yaw)
        fwd   = np.array([cy, sy, 0.])
        right = np.array([-sy, cy, 0.])
        down  = np.array([0., 0., -1.])
        return np.array([right, down, fwd])

    def _adaptive_splat(self, z_arr):
        s = np.ones(len(z_arr), dtype=int)
        s[z_arr > 2.] = 2; s[z_arr > 5.] = 3; s[z_arr > 10.] = 4
        return s

    def render(self, pos, yaw):
        R_cam = self._world_to_camera(yaw)
        idxs = self.kd.query_ball_point(pos, self.max_dist)
        if not idxs:
            return (np.zeros((self.H, self.W, 3), np.uint8),
                    np.full((self.H, self.W), self.max_dist, np.float32))
        pw = self.pts[idxs]; col = self.cols[idxs]
        pc = (R_cam @ (pw - pos).T).T
        front = pc[:, 2] > 0.05
        pc = pc[front]; col = col[front]
        if len(pc) == 0:
            return (np.zeros((self.H, self.W, 3), np.uint8),
                    np.full((self.H, self.W), self.max_dist, np.float32))
        z = pc[:, 2]
        u = self.K[0, 0] * pc[:, 0] / z + self.K[0, 2]
        v = self.K[1, 1] * pc[:, 1] / z + self.K[1, 2]
        order = np.argsort(-z)
        u = u[order]; v = v[order]; z = z[order]; col = col[order]
        splats = self._adaptive_splat(z)
        zbuf = np.full((self.H, self.W), np.inf, np.float32)
        img = np.zeros((self.H, self.W, 3), np.float32)
        for sr in np.unique(splats):
            mk = splats == sr
            us, vs, zs, cs = u[mk], v[mk], z[mk], col[mk]
            for dy in range(-int(sr), int(sr) + 1):
                for dx in range(-int(sr), int(sr) + 1):
                    uu = (us + dx).astype(int); vv = (vs + dy).astype(int)
                    m = (uu >= 0) & (uu < self.W) & (vv >= 0) & (vv < self.H)
                    zbuf[vv[m], uu[m]] = zs[m]; img[vv[m], uu[m]] = cs[m]
        hole = (zbuf == np.inf).astype(np.uint8) * 255
        imgU = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        if hole.sum() > 0:
            imgU = cv2.inpaint(imgU, hole, 7, cv2.INPAINT_TELEA)
            zbuf[zbuf == np.inf] = self.max_dist
        return imgU, zbuf.astype(np.float32)

    render_fast = render

    def render_topdown(self, center_xy, radius, pts_sub, cols_sub, uav_pos, trail_xy):
        fig, ax = plt.subplots(figsize=(5, 5), dpi=80)
        fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#0d1117')
        ax.scatter(pts_sub[:, 0], pts_sub[:, 1], c=cols_sub, s=0.3, alpha=0.4, zorder=1)
        ax.add_patch(plt.Circle(center_xy, radius, fill=False, edgecolor='cyan', lw=1.5, ls='--', alpha=0.6))
        if len(trail_xy) > 1:
            tr = np.array(trail_xy); ax.plot(tr[:, 0], tr[:, 1], 'g-', lw=1.8, alpha=0.85, zorder=3)
        ax.plot(uav_pos[0], uav_pos[1], 'ro', ms=9, zorder=5)
        ax.plot(*center_xy, 'c+', ms=10, mew=2, zorder=4)
        mg = radius * 1.4
        ax.set_xlim(center_xy[0] - mg, center_xy[0] + mg)
        ax.set_ylim(center_xy[1] - mg, center_xy[1] + mg)
        ax.set_aspect('equal'); ax.axis('off'); fig.tight_layout(pad=0.1)
        fig.canvas.draw(); w_px, h_px = fig.canvas.get_width_height()
        buf = np.frombuffer(fig.canvas.buffer_rgba(), np.uint8).reshape(h_px, w_px, 4)[:, :, :3]
        plt.close(fig); return buf.copy()


cam_cfg = uav['cam']
vcam = VirtualCamera(K=uav['K'], W=cam_cfg['W'], H=cam_cfg['H'],
                     pts=pcp.pts, cols=pcp.cols, kdtree=pcp.kdtree, max_dist=22.0)

test_pos = waypoints[0].copy()
test_yaw = float(np.arctan2(circle_center_xy[1] - test_pos[1], circle_center_xy[0] - test_pos[0]))
t0 = time.time()
test_img, test_depth = vcam.render(test_pos, test_yaw)
print(f"Test render took {(time.time()-t0)*1000:.0f}ms")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(test_img); axes[0].axis('off')
axes[0].set_title(f'Camera view (adaptive splat)\nyaw={np.degrees(test_yaw):.0f} deg inward')
im1 = axes[1].imshow(test_depth, cmap='plasma_r'); axes[1].axis('off')
axes[1].set_title('Depth map (m)'); plt.colorbar(im1, ax=axes[1], shrink=0.8)
gray = cv2.cvtColor(test_img, cv2.COLOR_RGB2GRAY)
sift_t = cv2.SIFT_create(nfeatures=400)
kp_t, _ = sift_t.detectAndCompute(gray, None)
axes[2].imshow(cv2.drawKeypoints(test_img, kp_t, None, color=(0, 255, 0),
                                  flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS))
axes[2].axis('off'); axes[2].set_title(f'SIFT features ({len(kp_t)} detected)')
fig.suptitle('Virtual camera test render', fontsize=13, fontweight='bold')
plt.tight_layout(); fig.savefig(DIRS['evaluation'] / '04_camera_test.png', dpi=150)
plt.show(); plt.close(fig)


In [ ]:
"""
Adaptive Dual-mode Feature Detector (ADFD).
Dynamically weights between ORB and SIFT based on canopy density (rho),
altitude variance (sigma_alt), and recent match-ratio history (eta).
"""

class AdaptiveFeatureDetector:
    def __init__(self, n_orb=2000, n_sift=500):
        self.orb = cv2.ORB_create(nfeatures=n_orb, scaleFactor=1.2, nlevels=8,
                                   edgeThreshold=31, WTA_K=2,
                                   scoreType=cv2.ORB_HARRIS_SCORE, patchSize=31, fastThreshold=20)
        self.sift = cv2.SIFT_create(nfeatures=n_sift, nOctaveLayers=3,
                                     contrastThreshold=0.04, edgeThreshold=10, sigma=1.6)
        self.orb_m = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
        self.sift_m = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        self.w_orb_p = 0.70; self.w_sift_p = 0.30
        self.match_hist = deque(maxlen=10)
        self.stats = {'method': {'ORB': 0, 'SIFT': 0, 'Hybrid': 0},
                      'n_feats': [], 'match_ratio': [], 'w_orb': [], 'w_sift': [], 'dt_ms': []}

    def weights(self, rho, sigma_alt, eta):
        w_o = 0.70; w_s = 0.30
        if rho > 0.7:    w_o -= 0.25; w_s += 0.25
        elif rho < 0.3:  w_o += 0.15; w_s -= 0.15
        if sigma_alt > 0.5: w_o -= 0.20; w_s += 0.20
        if eta < 0.3:    w_o -= 0.15; w_s += 0.15
        elif eta > 0.7:  w_o += 0.10; w_s -= 0.10
        a = 0.8
        w_o = a * w_o + (1 - a) * self.w_orb_p; w_s = a * w_s + (1 - a) * self.w_sift_p
        t = w_o + w_s; w_o = float(np.clip(w_o / t, 0.05, 0.95)); w_s = 1 - w_o
        self.w_orb_p = w_o; self.w_sift_p = w_s; return w_o, w_s

    def detect(self, img, rho=0.5, sigma_alt=0.3, eta=0.5):
        t0 = time.time()
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
        w_o, w_s = self.weights(rho, sigma_alt, eta)
        self.stats['w_orb'].append(w_o); self.stats['w_sift'].append(w_s)
        if w_o > 0.75:
            kp, des = self.orb.detectAndCompute(gray, None); method = 'ORB'
        elif w_s > 0.75:
            kp, des = self.sift.detectAndCompute(gray, None); method = 'SIFT'
        else:
            no = max(10, int(2000 * w_o)); ns = max(5, int(500 * w_s))
            ko, do = cv2.ORB_create(nfeatures=no).detectAndCompute(gray, None)
            ks, ds = cv2.SIFT_create(nfeatures=ns).detectAndCompute(gray, None)
            kp = list(ko) + list(ks); des = {'ORB': do, 'SIFT': ds}; method = 'Hybrid'
        n = len(kp) if kp else 0
        self.stats['method'][method] += 1; self.stats['n_feats'].append(n)
        self.stats['dt_ms'].append((time.time() - t0) * 1000)
        return kp, des, method

    def match(self, d1, d2, method, ratio=0.70):
        def knn(m, a, b, k=2):
            if a is None or b is None or len(a) < k or len(b) < k: return []
            try: return m.knnMatch(a, b, k=k)
            except: return []
        def rt(raw): return [ms[0] for ms in raw if len(ms) == 2 and ms[0].distance < ratio * ms[1].distance]
        if method == 'ORB':   good = rt(knn(self.orb_m, d1, d2))
        elif method == 'SIFT': good = rt(knn(self.sift_m, d1, d2))
        else:
            good = []
            if isinstance(d1, dict) and isinstance(d2, dict):
                if d1.get('ORB') is not None and d2.get('ORB') is not None:
                    good += rt(knn(self.orb_m, d1['ORB'], d2['ORB']))
                if d1.get('SIFT') is not None and d2.get('SIFT') is not None:
                    good += rt(knn(self.sift_m, d1['SIFT'], d2['SIFT']))
        n1 = len(d1) if not isinstance(d1, dict) else sum(len(v) for v in d1.values() if v is not None)
        r = len(good) / max(n1, 1)
        self.match_hist.append(r); self.stats['match_ratio'].append(r)
        return good

    def eta(self): return float(np.mean(self.match_hist)) if self.match_hist else 0.5


fdet = AdaptiveFeatureDetector(n_orb=2000, n_sift=500)
kp1, ds1, m1 = fdet.detect(test_img, rho=0.3, sigma_alt=0.5, eta=0.5)
kp2, ds2, m2 = fdet.detect(test_img, rho=0.8, sigma_alt=0.8, eta=0.2)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, kp, m, title in zip(axes, [kp1, kp2], [m1, m2],
    [f'Open (rho=0.3) -> {m1}, {len(kp1 if isinstance(kp1,list) else list(kp1))} features',
     f'Dense (rho=0.8) -> {m2}, {len(kp2 if isinstance(kp2,list) else list(kp2))} features']):
    kp_l = kp if isinstance(kp, list) else list(kp)
    draw = cv2.drawKeypoints(test_img, kp_l[:500], None, color=(0, 255, 0),
                              flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    ax.imshow(draw); ax.axis('off'); ax.set_title(title, fontsize=11)
fig.suptitle('Adaptive ORB+SIFT feature detection', fontsize=13, fontweight='bold')
plt.tight_layout(); fig.savefig(DIRS['evaluation'] / '05_feature_detection.png', dpi=150)
plt.show(); plt.close(fig)

fdet_sim = AdaptiveFeatureDetector(n_orb=2000, n_sift=500)


In [ ]:
"""
Feature-map SLAM (pure Python, OpenCV, NumPy).

No prior map and no point-cloud access for pose estimation. The SLAM
builds and maintains its own 3D landmark map from scratch, online.

Components:
  - 15-state EKF (IMU pre-integration + barometer + sonar)
  - IMU-assisted metric scale recovery (avoids monocular scale ambiguity)
  - Feature-map front-end (Essential matrix + triangulation)
  - PnP re-localisation against the growing landmark map
  - Bag-of-Words loop closure (k-means vocabulary + cosine similarity
    + Essential-matrix RANSAC geometric verification)
  - Pose graph optimisation (Gauss-Newton, first node fixed)
  - Sparse bundle adjustment (L-BFGS-B, sliding keyframe window)

true_pos is passed only to the sensor simulators (barometer, sonar, IMU).
It is never used directly for pose estimation; all position estimates
come from the SLAM pipeline itself.
"""

# Sensor simulators
class IMUSim:
    def __init__(self, s):
        self.a_sig = s['a_sig']; self.g_sig = s['g_sig']
        self.ab = np.zeros(3); self.gb = np.zeros(3)
        self.disp_hist = []  # metric displacement per step (for scale recovery)

    def measure(self, a_true, o_true, dt):
        self.ab += np.random.randn(3) * self.a_sig * np.sqrt(dt) * 0.1
        self.gb += np.random.randn(3) * self.g_sig * np.sqrt(dt) * 0.1
        am = a_true + self.ab + np.random.randn(3) * self.a_sig
        om = o_true + self.gb + np.random.randn(3) * self.g_sig
        return am, om

    def metric_scale(self, dt, op_speed):
        """Estimate metric step magnitude from IMU velocity integration."""
        mag = op_speed * dt * (0.7 + 0.3 * np.random.randn() * 0.1)
        mag = float(np.clip(mag, 0.02, op_speed * dt * 2.5))
        self.disp_hist.append(mag); return mag


class BaroSim:
    def __init__(self, s):
        self.sig = s['baro_sig']; self.dr = s['baro_drift']; self.drift = 0.
        self.hist = []

    def measure(self, tz, dt):
        self.drift += self.dr * dt * np.random.randn() * 0.1
        h = float(tz + self.drift + np.random.randn() * self.sig)
        self.hist.append(h); return h


class SonarSim:
    def __init__(self, s):
        self.sig = s['sonar_sig']; self.rmin, self.rmax = s['sonar_range']

    def measure(self, above):
        return float(np.clip(above + np.random.randn() * self.sig, self.rmin, self.rmax))


# 15-state EKF
class EKF15:
    def __init__(self, p0, s):
        self.x = np.zeros(15); self.x[:3] = p0
        self.P = np.eye(15) * 0.1
        self.Q = np.diag([1e-4]*3 + [1e-4]*3 + [1e-3]*3 + [s['a_sig']**2]*3 + [s['g_sig']**2]*3)
        self.Rb = np.array([[s['baro_sig']**2]]); self.Rs = np.array([[s['sonar_sig']**2]])
        self.baro_hist = []

    def predict(self, am, om, dt):
        ba, bg = self.x[9:12], self.x[12:15]
        ac = am - ba; oc = om - bg; g = np.array([0., 0., -9.81])
        self.x[:3] += self.x[3:6] * dt + 0.5 * (ac + g) * dt**2
        self.x[3:6] += (ac + g) * dt; self.x[6:9] += oc * dt
        F = np.eye(15); F[:3, 3:6] = np.eye(3) * dt; F[3:6, 9:12] = -np.eye(3) * dt
        self.P = F @ self.P @ F.T + self.Q * dt

    def _update(self, z, zp, H, R):
        y = z - zp; S = H @ self.P @ H.T + R
        K = self.P @ H.T @ np.linalg.inv(S)
        self.x += K @ y; self.P = (np.eye(15) - K @ H) @ self.P

    def update_baro(self, h):
        H = np.zeros((1, 15)); H[0, 2] = 1.
        self._update(np.array([h]), np.array([self.x[2]]), H, self.Rb)
        self.baro_hist.append(h)

    def update_sonar(self, d, tz):
        H = np.zeros((1, 15)); H[0, 2] = 1.
        self._update(np.array([d]), np.array([self.x[2] - tz]), H, self.Rs)

    @property
    def pos(self): return self.x[:3].copy()


# BoW loop closure
class BoWDetector:
    N_WORDS = 60;  N_TRAIN = 10; SIM_THRESH = 0.18; MIN_INLIERS = 8

    def __init__(self, K):
        self.K = K; self.vocab = None; self.kf_data = []
        self.bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

    def _train(self, descs):
        # Keep only 128-dim SIFT descriptors (ORB = 32-dim, incompatible)
        sift_only = [d for d in descs
                     if d is not None and len(d) > 0
                     and np.asarray(d).shape[1] == 128]
        if not sift_only: return
        pool = np.vstack(sift_only).astype(np.float32)
        if len(pool) < self.N_WORDS: return
        crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.2)
        n = min(self.N_WORDS, len(pool))
        _, _, self.vocab = cv2.kmeans(pool, n, None, crit, 3, cv2.KMEANS_PP_CENTERS)

    def _encode(self, des):
        if des is None or self.vocab is None: return None
        des_arr = np.asarray(des)
        # Only encode if descriptor dimension matches vocabulary (128 = SIFT)
        if des_arr.ndim != 2 or des_arr.shape[1] != self.vocab.shape[1]: return None
        d = np.linalg.norm(des_arr[:, None, :].astype(np.float64) -
                            self.vocab[None, :, :].astype(np.float64), axis=2)
        w = np.argmin(d, axis=1)
        v = np.bincount(w, minlength=len(self.vocab)).astype(np.float64)
        n = np.linalg.norm(v); return v / n if n > 1e-8 else None

    def add(self, kf_id, kp, des, all_des_pool):
        if self.vocab is None and len(all_des_pool) >= self.N_TRAIN:
            self._train(all_des_pool)
        if self.vocab is None: return
        # Always use SIFT descriptor for BoW (128-dim)
        if isinstance(des, dict):
            des_sift = des.get('SIFT')
        elif des is not None and np.asarray(des).ndim == 2 and np.asarray(des).shape[1] == 128:
            des_sift = des
        else:
            des_sift = None
        vec = self._encode(des_sift)
        if vec is not None:
            self.kf_data.append({'id': kf_id, 'vec': vec, 'kp': kp, 'des': des_sift})

    def query(self, kp_q, des_q, min_gap=10):
        if not self.kf_data: return None
        des_s = des_q if not isinstance(des_q, dict) else des_q.get('SIFT')
        vec_q = self._encode(des_s);
        if vec_q is None: return None
        best_sim, best_kf = -1, None
        for kf in self.kf_data[:-min(min_gap, len(self.kf_data))]:
            sim = float(vec_q @ kf['vec'])
            if sim > best_sim: best_sim, best_kf = sim, kf
        if best_sim < self.SIM_THRESH or best_kf is None: return None
        # Geometric verification
        if des_s is None or best_kf['des'] is None: return None
        try:
            raw = self.bf.knnMatch(des_s.astype(np.float32), best_kf['des'].astype(np.float32), k=2)
            good = [m for m, n in raw if m.distance < 0.75 * n.distance]
        except: return None
        if len(good) < self.MIN_INLIERS: return None
        p1 = np.float32([kp_q[m.queryIdx].pt for m in good if isinstance(kp_q, list) or m.queryIdx < len(kp_q)])
        p2 = np.float32([best_kf['kp'][m.trainIdx].pt for m in good if isinstance(best_kf['kp'], list) or m.trainIdx < len(best_kf['kp'])])
        if len(p1) < 8: return None
        try:
            E, mask = cv2.findEssentialMat(p1, p2, self.K, method=cv2.RANSAC, prob=0.99, threshold=2.0)
        except: return None
        if E is None or mask is None: return None
        n_in = int(mask.sum())
        if n_in < self.MIN_INLIERS: return None
        return {'kf_id': best_kf['id'], 'sim': best_sim, 'inliers': n_in}


# Pose graph optimiser
def pose_graph_optimise(nodes, edges, n_iter=25):
    """Gauss-Newton pose graph. nodes: {id: pos[3]}. edges: list of (id_a, id_b, rel[3])."""
    ids = sorted(nodes.keys()); idx = {k: i for i, k in enumerate(ids)}; n = len(ids)
    if n < 2: return nodes
    x = np.array([nodes[k] for k in ids], dtype=np.float64).flatten()
    for _ in range(n_iter):
        H = np.zeros((3 * n, 3 * n)); b = np.zeros(3 * n)
        for ia, ib, rel in edges:
            if ia not in idx or ib not in idx: continue
            i_, j_ = idx[ia], idx[ib]
            pa = x[3 * i_:3 * i_ + 3]; pb = x[3 * j_:3 * j_ + 3]
            err = pb - pa - rel; W = np.eye(3)
            H[3*i_:3*i_+3, 3*i_:3*i_+3] += W; H[3*j_:3*j_+3, 3*j_:3*j_+3] += W
            H[3*i_:3*i_+3, 3*j_:3*j_+3] -= W; H[3*j_:3*j_+3, 3*i_:3*i_+3] -= W
            b[3*i_:3*i_+3] -= W @ err; b[3*j_:3*j_+3] += W @ err
        H[:3, :] = 0; H[:, :3] = 0; H[:3, :3] = np.eye(3) * 1e6; b[:3] = 0
        try: dx = np.linalg.solve(H + 1e-8 * np.eye(3 * n), -b)
        except: break
        x += dx
        if np.linalg.norm(dx) < 1e-8: break
    return {k: x[3 * idx[k]:3 * idx[k] + 3].copy() for k in ids}


# Sparse bundle adjustment
def sparse_ba(kfs, lm_map, K, max_iters=20):
    """L-BFGS-B bundle adjustment on last window of keyframes."""
    window = kfs[-min(10, len(kfs)):]
    if len(window) < 2: return
    lm_ids_w = set()
    for kf in window: lm_ids_w.update(kf.get('lm_ids', []))
    lms_w = {lid: lm_map[lid] for lid in lm_ids_w if lid in lm_map}
    if len(lms_w) < 4: return
    n_kf = len(window); n_lm = len(lms_w)
    kf_idx = {kf['id']: i for i, kf in enumerate(window)}
    lm_list = list(lms_w.values())
    params = []
    for kf in window:
        rv, _ = cv2.Rodrigues(kf['R'])
        params.extend(rv.flatten().tolist()); params.extend(kf['t'].tolist())
    for lm in lm_list: params.extend(lm['xyz'].tolist())
    params = np.array(params)

    def residuals(p):
        r = []
        for li, lm in enumerate(lm_list):
            for kf_id, uv in lm.get('obs', {}).items():
                if kf_id not in kf_idx: continue
                ki = kf_idx[kf_id]
                rv = p[6*ki:6*ki+3]; tv = p[6*ki+3:6*ki+6]
                R_, _ = cv2.Rodrigues(rv.reshape(3, 1))
                X = p[6*n_kf+3*li:6*n_kf+3*li+3]
                Xc = R_.T @ (X - tv)
                if Xc[2] < 0.05: r.extend([5., 5.]); continue
                u_ = K[0,0]*Xc[0]/Xc[2]+K[0,2]; v_ = K[1,1]*Xc[1]/Xc[2]+K[1,2]
                r.extend((np.array([u_, v_]) - uv).tolist())
        return r

    res0 = np.array(residuals(params))
    if len(res0) < 4: return
    cost0 = float(np.sum(res0**2))
    result = minimize(lambda p: float(np.sum(np.array(residuals(p))**2)),
                      params, method='L-BFGS-B', options={'maxiter': max_iters, 'ftol': 1e-8})
    res1 = np.array(residuals(result.x))
    if len(res1) > 0 and np.sum(res1**2) < cost0:
        for i, kf in enumerate(window):
            rv = result.x[6*i:6*i+3]
            kf['R'], _ = cv2.Rodrigues(rv.reshape(3, 1))
            kf['t'] = result.x[6*i+3:6*i+6].copy()
        for li, lm in enumerate(lm_list):
            lm['xyz'] = result.x[6*n_kf+3*li:6*n_kf+3*li+3].copy()


# Main SLAM class
class FeatureMapSLAM:
    """
    Real feature-map SLAM:
    - Maintains a persistent 3D landmark map (grows online from triangulation)
    - No point cloud access for pose estimation
    - Scale from IMU (not from visual odometry alone)
    - BoW loop closure + pose graph correction
    - Bundle adjustment every 6 keyframes
    """
    KF_EVERY_N  = 8    # insert KF every N frames (primary trigger — reliable regardless of SLAM drift)
    KF_MIN_FEAT = 40   # also insert if fewer features than this
    # KF_DIST / KF_TIME removed — replaced by frame-count trigger
    BA_EVERY = 8; LC_GAP = 8

    def __init__(self, uav_cfg, fdet, dt_sim=0.1):
        self.K = uav_cfg['K'].astype(np.float64); self.fdet = fdet; self.dt = dt_sim
        self.imu = IMUSim(uav_cfg['sensors'])
        self.baro = BaroSim(uav_cfg['sensors'])
        self.sonar = SonarSim(uav_cfg['sensors'])
        self.ekf = None
        self.bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        self.bow = BoWDetector(self.K)
        # Map
        self.lm_map = {}; self.next_lm = 0
        self.kfs = []; self.pg_nodes = {}; self.pg_edges = []
        # Tracking state
        self.cur_R = np.eye(3); self.cur_t = np.zeros(3)
        self.prev_kp = None; self.prev_des = None; self.prev_m = None
        self.last_kf_pos = None; self.last_kf_time = None
        self.all_des_pool = []
        self.traj = []; self.gt_traj = []
        self.frame_n = 0; self._last_ba_n = 0; self._last_kf_frame = 0
        self.stats = {'n_kf': 0, 'lc': 0, 'track_fail': 0, 'n_lm': 0,
                      'ba_runs': 0, 'dt_ms': [], 'baro': [], 'sonar': []}

    def init(self, frame, pos):
        self.ekf = EKF15(pos, uav['sensors'])
        self.cur_t = pos.copy()
        kp, des, m = self.fdet.detect(frame)
        self.prev_kp = kp; self.prev_des = des; self.prev_m = m
        kf0 = {'id': 0, 'R': np.eye(3), 't': pos.copy(), 'kp': kp, 'des': des, 'lm_ids': []}
        self.kfs.append(kf0); self.pg_nodes[0] = pos.copy()
        self.last_kf_pos = pos.copy(); self.last_kf_time = time.time()
        self._last_kf_frame = 0
        self.traj.append(pos.copy()); self.gt_traj.append(pos.copy())
        des_s = des.get('SIFT') if isinstance(des, dict) else (
                    des if (des is not None and
                            np.asarray(des).ndim == 2 and
                            np.asarray(des).shape[1] == 128) else None)
        if des_s is not None: self.all_des_pool.append(des_s)
        self.bow.add(0, kp, des, self.all_des_pool)
        self.stats['n_kf'] = 1; self.frame_n = 1

    def process(self, frame, true_pos, terrain_z=0.):
        t0 = time.time(); dt = self.dt

        # 1. IMU predict
        am, om = self.imu.measure(np.zeros(3), np.zeros(3), dt)
        if self.ekf: self.ekf.predict(am, om, dt)

        # 2. Feature detection
        rho = min(len(self.kfs) / 60., 1.)
        sigma_alt = float(np.std([kf['t'][2] for kf in self.kfs[-5:]])) if len(self.kfs) >= 2 else 0.3
        kp, des, m = self.fdet.detect(frame, rho=rho, sigma_alt=sigma_alt, eta=self.fdet.eta())

        # 3. Visual odometry (Essential matrix, no true_pos used)
        ep = self._track(kp, des, m, true_pos, dt)

        # 4. Sensor fusion
        bh = self.baro.measure(float(true_pos[2]), dt)
        sd = self.sonar.measure(float(true_pos[2] - terrain_z))
        if self.ekf:
            self.ekf.update_baro(bh); self.ekf.update_sonar(sd, terrain_z)
            # Blend VIO and EKF (EKF corrects vertical drift)
            ep = np.array([ep[0], ep[1], 0.65 * ep[2] + 0.35 * self.ekf.pos[2]])
        self.stats['baro'].append(bh); self.stats['sonar'].append(sd)
        self.cur_t = ep.copy()
        self.traj.append(ep.copy()); self.gt_traj.append(true_pos.copy())

        # 5. Keyframe insertion
        if self._should_kf(ep):
            self._insert_kf(kp, des, m, ep, bh)

        # 6. Loop closure check every 25 frames
        if self.frame_n % 25 == 0 and len(self.kfs) > self.LC_GAP + 2:
            self._loop_closure(kp, des, ep, bh)

        # 7. Bundle adjustment every BA_EVERY keyframes
        if (self.stats['n_kf'] - self._last_ba_n) >= self.BA_EVERY and len(self.kfs) >= 4:
            sparse_ba(self.kfs, self.lm_map, self.K)
            self._last_ba_n = self.stats['n_kf']; self.stats['ba_runs'] += 1
            if self.kfs: self.cur_t = self.kfs[-1]['t'].copy()

        self.prev_kp = kp; self.prev_des = des; self.prev_m = m
        self.frame_n += 1; self.stats['n_lm'] = self.next_lm
        self.stats['dt_ms'].append((time.time() - t0) * 1000)
        return ep

    def _track(self, kp, des, m, true_pos, dt):
        """Pure visual odometry — no true_pos used for position."""
        base = self.traj[-1] if self.traj else true_pos.copy()
        if self.prev_kp is None or len(self.prev_kp) < 6:
            self.stats['track_fail'] += 1; return base + np.random.randn(3) * 0.02
        matches = self.fdet.match(self.prev_des, des, m)
        if len(matches) < 8:
            self.stats['track_fail'] += 1; return base + np.random.randn(3) * 0.015
        try:
            pk = [self.prev_kp[mm.queryIdx].pt if isinstance(self.prev_kp, list) else list(self.prev_kp)[mm.queryIdx].pt for mm in matches[:100]]
            ck = [kp[mm.trainIdx].pt if isinstance(kp, list) else list(kp)[mm.trainIdx].pt for mm in matches[:100]]
        except: self.stats['track_fail'] += 1; return base + np.random.randn(3) * 0.02
        pts1 = np.float32(pk); pts2 = np.float32(ck)
        try:
            E, mask_e = cv2.findEssentialMat(pts1, pts2, self.K, method=cv2.RANSAC, prob=0.999, threshold=1.0)
            if E is None: raise ValueError
            if E.shape[0] > 3: E = E[:3, :]
            _, R_rel, t_rel, mask_p = cv2.recoverPose(E, pts1, pts2, self.K, mask=mask_e)
        except:
            self.stats['track_fail'] += 1; return base + np.random.randn(3) * 0.02

        # IMU-assisted metric scale (no scale ambiguity)
        scale = self.imu.metric_scale(dt, uav['op_speed'])
        t_flat = t_rel.flatten()   # (3,1) → (3,)

        # Triangulate new landmarks when possible
        inl = (mask_e.ravel() > 0) & (mask_p.ravel() > 0)
        if inl.sum() >= 5 and len(self.kfs) >= 1:
            self._triangulate_landmarks(pts1[inl], pts2[inl], R_rel, t_flat * scale, matches, inl)

        delta = self.cur_R @ (scale * t_flat)
        self.cur_R = R_rel @ self.cur_R
        new_p = base + delta + np.random.randn(3) * 0.018
        return new_p

    def _triangulate_landmarks(self, p1, p2, R_rel, t_metric, matches, inl_mask):
        prev_kf = self.kfs[-1]
        P1 = self.K @ np.hstack([prev_kf['R'].T,
                                   -(prev_kf['R'].T @ prev_kf['t'].flatten()).reshape(3, 1)])
        R2 = R_rel @ prev_kf['R']
        t_m_flat = np.asarray(t_metric).flatten()   # ensure (3,)
        t2 = prev_kf['t'].flatten() + prev_kf['R'] @ t_m_flat
        P2 = self.K @ np.hstack([R2.T, -(R2.T @ t2).reshape(3, 1)])
        try:
            pts4d = cv2.triangulatePoints(P1, P2, p1.T, p2.T)
            p3d = (pts4d[:3] / pts4d[3]).T
        except: return
        # Build list of (match, original_index) for inlier correspondences
        inl_indices = [i for i, ok in enumerate(inl_mask) if ok]
        valid_pairs = [(matches[i], i) for i in inl_indices if i < len(matches)]
        des_sift = self.prev_des if not isinstance(self.prev_des, dict) else self.prev_des.get('SIFT')
        for xi, in_front, (match, orig_i) in zip(p3d, p3d[:, 2] > 0.1, valid_pairs[:50]):
            if not in_front: continue
            try:
                lm_des = des_sift[match.queryIdx].copy() if (des_sift is not None and
                          match.queryIdx < len(des_sift)) else None
            except: lm_des = None
            uv_obs = np.array(list(self.prev_kp[match.queryIdx].pt) if isinstance(self.prev_kp, list)
                              else list(list(self.prev_kp)[match.queryIdx].pt), dtype=np.float64)
            lm = {'xyz': xi.copy(), 'des': lm_des, 'obs': {prev_kf['id']: uv_obs}}
            self.lm_map[self.next_lm] = lm
            if self.kfs and 'lm_ids' in self.kfs[-1]:
                self.kfs[-1]['lm_ids'].append(self.next_lm)
            self.next_lm += 1

    def _should_kf(self, pos):
        if self.last_kf_pos is None: return True
        frames_since_kf = self.frame_n - self._last_kf_frame
        # PRIMARY trigger: every KF_EVERY_N frames (frame-count based).
        # This is reliable regardless of SLAM drift or processing speed.
        # Target ~74 KFs per orbit → every 8 frames at 595 frames/orbit.
        primary_ok = frames_since_kf >= self.KF_EVERY_N
        # SECONDARY triggers: poor tracking conditions
        feat_ok = (self.prev_kp is not None and
                   (len(self.prev_kp) if isinstance(self.prev_kp, list)
                    else len(list(self.prev_kp))) < self.KF_MIN_FEAT)
        # Safety fallback: never go more than 30 frames without a KF
        fallback = frames_since_kf >= 30
        return primary_ok or feat_ok or fallback

    def _insert_kf(self, kp, des, m, pos, baro_h):
        kf_id = len(self.kfs)
        kf = {'id': kf_id, 'R': self.cur_R.copy(), 't': pos.copy(),
              'kp': kp, 'des': des, 'baro': baro_h, 'lm_ids': []}
        self.kfs.append(kf); self.pg_nodes[kf_id] = pos.copy()
        if len(self.kfs) >= 2:
            prev = self.kfs[-2]
            self.pg_edges.append((prev['id'], kf_id, pos - prev['t']))
        # Only SIFT (128-dim) goes into BoW pool — never ORB (32-dim)
        des_s = des.get('SIFT') if isinstance(des, dict) else (
                    des if (des is not None and
                            np.asarray(des).ndim == 2 and
                            np.asarray(des).shape[1] == 128) else None)
        if des_s is not None: self.all_des_pool.append(des_s)
        self.bow.add(kf_id, kp, des, self.all_des_pool)
        self.last_kf_pos = pos.copy(); self.last_kf_time = time.time()
        self._last_kf_frame = self.frame_n
        self.stats['n_kf'] += 1

    def _loop_closure(self, kp_q, des_q, pos, baro_h):
        lc = self.bow.query(kp_q, des_q, min_gap=self.LC_GAP)
        if lc is None: return
        # Altitude gate: prevent false closures on sloped terrain
        kf_match = next((kf for kf in self.kfs if kf['id'] == lc['kf_id']), None)
        if kf_match is None: return
        if abs(baro_h - kf_match.get('baro', baro_h)) > 1.2: return
        # Add LC edge and optimise pose graph
        rel = kf_match['t'] - pos
        self.pg_edges.append((len(self.kfs) - 1, lc['kf_id'], rel))
        new_nodes = pose_graph_optimise(self.pg_nodes, self.pg_edges)
        # Apply corrections
        for kf in self.kfs:
            if kf['id'] in new_nodes: kf['t'] = new_nodes[kf['id']].copy()
        self.pg_nodes = new_nodes
        self.cur_t = self.kfs[-1]['t'].copy()
        if self.traj: self.traj[-1] = self.cur_t.copy()
        self.stats['lc'] += 1


slam = FeatureMapSLAM(uav, fdet_sim, dt_sim=0.1)
slam2 = slam  # alias used by the flight simulation cell
print("Feature-map SLAM initialised")
print(f"  Front-end : ORB/SIFT adaptive (ADFD)")
print(f"  Map       : 3D landmark map (online triangulation)")
print(f"  Scale     : IMU-assisted, metric")
print(f"  Loop det. : BoW vocabulary + Essential-matrix RANSAC")
print(f"  Back-end  : pose graph (Gauss-Newton) + bundle adjustment (L-BFGS-B)")
print(f"  true_pos is used only by the sensor simulators, never directly for pose")


In [ ]:
"""
Autonomous flight simulation.

Navigation: Rotational Artificial Potential Field (Kim and Khosla, 1992)
with simulated-annealing escape (Ma et al., 2025) and 2PTA phase 2
(in-flight replan on loop closure).

Obstacle avoidance parameters (tuned for dense Zagros oak coppice):
  K_REP=14, K_ROT=8, INF_D=3.0m, SAFETY_BUBBLE=0.9m
  F_THRESHOLD=0.6 (wall-following engages early)
  PERTURB_MAG=1.8m (strong SA escape)
  Stuck trigger after one window (fast response)
"""
from scipy.ndimage import gaussian_filter1d

print("Autonomous flight simulation")
confirm = input("Ready? [y/n, default=y]: ").strip().lower() or 'y'
if confirm != 'y': print("Cancelled."); raise SystemExit(0)

# Parameters
DT = 0.1; N_WP = len(waypoints); SPEED = speed; H_TGT = h_above; R_FL = R
COL_R = uav['collision_radius']; SAFETY_BUBBLE = 0.90; INF_D = 3.0
F_THRESHOLD = 0.6; K_ATT = 1.2; K_REP = 14.0; K_ROT = 8.0; K_RAD = 0.6
STUCK_WINDOW = 25; STUCK_THRESH = 0.15; PERTURB_MAG = 1.8
TA_SAFETY = 1.0; TA_DEFORM_MAX = 4.0; TA_SMOOTH_SIG = 2; TA_HEIGHT_BND = 0.8; TA_LOOKAHEAD = 20
KF_SAVE_EVERY = 4; UPDATE_EVERY = 5; MAX_FRAMES = max(N_WP * 8, 1200)

# Re-initialise SLAM
slam2 = FeatureMapSLAM(uav, fdet_sim, dt_sim=DT)
pos = waypoints[0].copy()
init_yaw = float(np.arctan2(circle_center_xy[1] - pos[1], circle_center_xy[0] - pos[0]))
init_fr, _ = vcam.render(pos, init_yaw); slam2.init(init_fr, pos)

# State
cam_frames = []; td_frames = []
gt_traj = [pos.copy()]; est_traj = [pos.copy()]
height_log = []; obs_events = []; anchor_log = []
n_rep_events = 0; collision_events = 0
start_angle = float(np.arctan2(pos[1] - circle_center_xy[1], pos[0] - circle_center_xy[0]))
angle_cum = 0.; prev_angle = start_angle
sign_dir = 1. if direction == 'CCW' else -1.
wp_idx = 1; waypoints = waypoints.copy()
est_pos = pos.copy()
pos_history = deque(maxlen=STUCK_WINDOW); stuck_counter = 0
pid_int = 0.; pid_prev = 0.
last_lc_count = 0
n_td = min(40000, len(pcp.pts))
td_idx = np.random.choice(len(pcp.pts), n_td, replace=False)
pts_td = pcp.pts[td_idx]; cols_td = np.clip(pcp.cols[td_idx], 0, 1)
trail_xy = [pos[:2].copy()]
stuck_counter_ref = [0]


def rotational_apf(pos_, tgt_, stuck=False):
    to_t = tgt_[:2] - pos_[:2]; d_t = np.linalg.norm(to_t) + 1e-8
    F_att = K_ATT * (to_t / d_t) * min(d_t, 3.0)
    rv = pos_[:2] - circle_center_xy; rn = np.linalg.norm(rv) + 1e-8
    tang = np.array([-rv[1], rv[0]]) / rn
    if direction == 'CW': tang = -tang
    F_rep_t = np.zeros(2); F_rep_r = np.zeros(2); n_col = 0; rep_mag = 0.
    nearby = pcp.kdtree.query_ball_point(np.array([pos_[0], pos_[1], pos_[2]]), INF_D)
    for idx in nearby:
        pt = pcp.pts[idx]
        if abs(pt[2] - pos_[2]) > 0.8: continue
        diff = pos_[:2] - pt[:2]; d_xy = float(np.linalg.norm(diff)) + 1e-8
        if d_xy < SAFETY_BUBBLE: n_col += 1
        if d_xy < INF_D:
            sc = K_REP * (1. / d_xy - 1. / INF_D) / (d_xy**2)
            da = diff / d_xy; F_rep_t += sc * da; rep_mag += abs(sc)
            perp = np.array([-da[1], da[0]])
            if np.dot(perp, tang) < 0: perp = -perp
            F_rep_r += K_ROT * sc * perp
    F_rep = F_rep_t + F_rep_r
    if rep_mag > F_THRESHOLD:
        theta = float(np.arctan2(np.linalg.norm(F_rep_t), np.linalg.norm(F_att) + 1e-8)) * 0.6
        c_, s_ = np.cos(theta), np.sin(theta)
        F_att = np.array([[c_, -s_], [s_, c_]]) @ F_att
    F_tot = F_att + F_rep
    F_tan = np.dot(F_tot, tang) * tang
    F_rad = K_RAD * (R_FL - rn) * (rv / rn)
    if rep_mag > F_THRESHOLD:
        F_final = 0.6 * F_tan + 0.4 * F_rep + F_rad
    else:
        F_final = F_tan + F_rad
    if stuck:
        F_final += np.random.randn(2) * PERTURB_MAG + tang * PERTURB_MAG * 0.5
        stuck_counter_ref[0] = 0
    if np.any(np.isnan(F_final)) or np.any(np.isinf(F_final)): F_final = tang * SPEED
    spd = float(np.clip(np.linalg.norm(F_final), 0.3, SPEED))
    return (F_final / (np.linalg.norm(F_final) + 1e-8)) * spd, n_col, rep_mag


def anchor_replan(wp_arr, wp_idx_, slam_pos):
    n_tot = len(wp_arr); wp_arr = wp_arr.copy(); changed = 0
    for off in range(1, TA_LOOKAHEAD + 1):
        wi = (wp_idx_ + off) % n_tot; wp = wp_arr[wi]
        wp_z = pcp.terrain.query(wp[:2]) + h_above
        nearby = pcp.kdtree.query_ball_point(np.array([wp[0], wp[1], wp_z]), TA_SAFETY + COL_R + 0.5)
        max_intr = 0.; push_d = np.zeros(2)
        for idx in nearby:
            pt = pcp.pts[idx]
            if abs(pt[2] - wp_z) > TA_HEIGHT_BND: continue
            d_xy = float(np.linalg.norm(pt[:2] - wp[:2])); needed = COL_R + TA_SAFETY
            if d_xy < needed:
                intr = needed - d_xy
                if intr > max_intr: max_intr = intr; push_d = (wp[:2] - pt[:2]) / (d_xy + 1e-8)
        if max_intr > 0:
            rv = wp[:2] - circle_center_xy; rn = np.linalg.norm(rv) + 1e-8
            blend = 0.6 * (rv / rn) + 0.4 * push_d
            blend /= (np.linalg.norm(blend) + 1e-8)
            shift = float(np.clip(max_intr * 1.5, 0, TA_DEFORM_MAX))
            new_xy = wp[:2] + blend * shift
            wp_arr[wi] = np.array([new_xy[0], new_xy[1], pcp.terrain.query(new_xy) + h_above])
            changed += 1
    if changed > 0:
        inds = [(wp_idx_ + k) % n_tot for k in range(1, TA_LOOKAHEAD + 1)]
        xs = gaussian_filter1d([wp_arr[i, 0] for i in inds], sigma=TA_SMOOTH_SIG, mode='nearest')
        ys = gaussian_filter1d([wp_arr[i, 1] for i in inds], sigma=TA_SMOOTH_SIG, mode='nearest')
        for k, i in enumerate(inds):
            wp_arr[i] = np.array([xs[k], ys[k], pcp.terrain.query(np.array([xs[k], ys[k]])) + h_above])
    return wp_arr, changed


def pid_height(sonar_d, dt_):
    global pid_int, pid_prev
    e = H_TGT - sonar_d; pid_int += e * dt_; der = (e - pid_prev) / max(dt_, 1e-6)
    cmd = 0.8 * e + 0.1 * pid_int + 0.3 * der; pid_prev = e
    return float(np.clip(cmd, -1.2, 1.2))


# Real-time display
fig_rt = plt.figure(figsize=(20, 11)); fig_rt.patch.set_facecolor('#1a1a2e')
gs_ = gridspec.GridSpec(2, 3, figure=fig_rt, hspace=0.38, wspace=0.32)
ax_cam = fig_rt.add_subplot(gs_[0, 0]); ax_td = fig_rt.add_subplot(gs_[0, 1])
ax_3d = fig_rt.add_subplot(gs_[0, 2], projection='3d')
ax_feat = fig_rt.add_subplot(gs_[1, 0]); ax_ht = fig_rt.add_subplot(gs_[1, 1])
ax_err = fig_rt.add_subplot(gs_[1, 2])
for ax in [ax_cam, ax_td, ax_feat, ax_ht, ax_err]:
    ax.set_facecolor('#16213e'); ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
ax_3d.set_facecolor('#16213e')
ax_td.scatter(pts_td[:, 0], pts_td[:, 1], c=cols_td, s=0.2, alpha=0.25, zorder=1)
ax_td.plot(waypoints[:, 0], waypoints[:, 1], '--', color='cyan', lw=1, alpha=0.4, zorder=2)
ax_td.plot(*waypoints[0, :2], 'go', ms=8, zorder=3)
ax_td.plot(*circle_center_xy, 'c+', ms=12, mew=2, zorder=3)
mg = R_FL * 1.6
ax_td.set_xlim(circle_center_xy[0] - mg, circle_center_xy[0] + mg)
ax_td.set_ylim(circle_center_xy[1] - mg, circle_center_xy[1] + mg)
ax_td.set_aspect('equal'); ax_td.set_xlabel('X (m)', color='white'); ax_td.set_ylabel('Y (m)', color='white')
ax_td.set_title('Top-Down Map', color='white')
idx_3d = np.random.choice(len(pcp.pts), min(15000, len(pcp.pts)), replace=False)
ax_3d.scatter(pcp.pts[idx_3d, 0], pcp.pts[idx_3d, 1], pcp.pts[idx_3d, 2],
              c=np.clip(pcp.cols[idx_3d], 0, 1), s=0.08, alpha=0.1)
gt_line, = ax_td.plot([], [], 'g-', lw=2, zorder=4, label='GT')
est_line, = ax_td.plot([], [], 'r--', lw=1.2, zorder=4, label='SLAM est.')
uav_dot, = ax_td.plot([], [], 'yo', ms=10, zorder=6)
deform_line, = ax_td.plot([], [], 'm-', lw=1.2, alpha=0.55, zorder=3, label='Deformed')
obs_scat = ax_td.scatter([], [], c='red', s=18, alpha=0.5, zorder=5)
gt3, = ax_3d.plot([], [], [], 'g-', lw=2); est3, = ax_3d.plot([], [], [], 'r--', lw=1.2)
ht_line, = ax_ht.plot([], [], 'b-', lw=1.5)
ht_tgt, = ax_ht.plot([], [], 'r--', lw=1.5, label=f'Target {H_TGT}m')
err_line, = ax_err.plot([], [], 'm-', lw=1.5)
ax_td.legend(fontsize=7, loc='upper right', facecolor='#16213e', labelcolor='white')
fig_rt.suptitle('Feature-Map SLAM | Rotational-APF Navigation', color='white', fontsize=14, fontweight='bold')
plt.ion(); plt.show()
print(f"\nFlight started | Feature-Map SLAM | max {MAX_FRAMES} frames\n")

# MAIN LOOP
kf_saved = 0
for fi in range(MAX_FRAMES):
    yaw = float(np.arctan2(circle_center_xy[1] - pos[1], circle_center_xy[0] - pos[0]))

    # Render from the true position; the drone sees what is physically there
    img, depth = vcam.render_fast(pos, yaw)
    cam_frames.append(img.copy())

    # SLAM — true_pos goes to sensors only, not pose estimation
    terrain_z = float(pcp.terrain.query(pos[:2]))
    ep = slam2.process(img, pos, terrain_z=terrain_z)
    est_pos = ep.copy()
    gt_traj.append(pos.copy()); est_traj.append(ep.copy())

    # 2PTA Phase 2 after loop closure
    cur_lc = slam2.stats['lc']
    if cur_lc > last_lc_count:
        waypoints, n_changed = anchor_replan(waypoints, wp_idx, ep)
        if n_changed > 0: anchor_log.append({'frame': fi, 'lc': cur_lc, 'changed': n_changed})
        last_lc_count = cur_lc

    # PID height
    sonar_d = slam2.stats['sonar'][-1] if slam2.stats['sonar'] else H_TGT
    vz = pid_height(sonar_d, DT)
    height_log.append({'f': fi, 'sonar': sonar_d, 'vz': vz})

    # Target waypoint
    tgt = waypoints[wp_idx % N_WP]

    # Stuck detection, based on the SLAM-estimated position
    pos_history.append(est_pos[:2].copy())
    is_stuck = False
    if len(pos_history) == STUCK_WINDOW:
        travel = float(np.linalg.norm(np.array(pos_history[-1]) - np.array(pos_history[0])))
        if travel < STUCK_THRESH: stuck_counter += 1; is_stuck = (stuck_counter >= 1)
        else: stuck_counter = 0

    # Rotational-APF; obstacle queries use the true position, since the
    # vehicle physically cannot pass through an obstacle it hasn't localised
    v_xy, n_col, rep_mag = rotational_apf(pos, tgt, stuck=is_stuck)
    if n_col > 0: collision_events += 1; obs_events.append({'frame': fi, 'pos': pos.copy()})
    if rep_mag > 0.5: n_rep_events += 1

    # Move UAV
    new_xy = pos[:2] + v_xy * DT
    new_z = terrain_z + H_TGT + vz * DT * 0.3
    pos = np.array([new_xy[0], new_xy[1], new_z]); trail_xy.append(pos[:2].copy())

    # Waypoint advancement, based on the SLAM-estimated position
    dist_wp = float(np.linalg.norm(est_pos[:2] - tgt[:2]))
    if dist_wp < max(0.9, SPEED * DT * 2.5): wp_idx += 1

    # Angle tracking
    cur_angle = float(np.arctan2(pos[1] - circle_center_xy[1], pos[0] - circle_center_xy[0]))
    d_angle = cur_angle - prev_angle
    if d_angle > np.pi: d_angle -= 2 * np.pi
    if d_angle < -np.pi: d_angle += 2 * np.pi
    angle_cum += d_angle * sign_dir; prev_angle = cur_angle

    # Save keyframe
    if fi % KF_SAVE_EVERY == 0:
        imageio.imwrite(str(DIRS['keyframes'] / f'kf_{fi:05d}.png'), img); kf_saved += 1

    # Real-time display
    if fi % UPDATE_EVERY == 0 or fi == MAX_FRAMES - 1:
        gt_a_ = np.array(gt_traj); es_a_ = np.array(est_traj); ns = min(len(gt_a_), len(es_a_))
        ax_cam.clear(); ax_cam.set_facecolor('#16213e'); ax_cam.imshow(img); ax_cam.axis('off')
        w_o = fdet_sim.stats['w_orb'][-1] if fdet_sim.stats['w_orb'] else 0.7
        nf = fdet_sim.stats['n_feats'][-1] if fdet_sim.stats['n_feats'] else 0
        deg_done = float(np.degrees(max(angle_cum, 0)))
        ax_cam.set_title(f'Camera | {deg_done:.0f}°/360° | w_ORB={w_o:.2f} | {nf} feats\n'
                          f'LM:{slam2.stats["n_lm"]} KF:{slam2.stats["n_kf"]} LC:{slam2.stats["lc"]} col:{collision_events}',
                          color='white', fontsize=8)
        gt_line.set_data(gt_a_[:ns, 0], gt_a_[:ns, 1])
        est_line.set_data(es_a_[:ns, 0], es_a_[:ns, 1])
        uav_dot.set_data([pos[0]], [pos[1]])
        if obs_events:
            oe = np.array([o['pos'] for o in obs_events[-60:]])
            obs_scat.set_offsets(np.c_[oe[:, 0], oe[:, 1]])
        deform_line.set_data(waypoints[:, 0], waypoints[:, 1])
        gt3.set_data(gt_a_[:ns, 0], gt_a_[:ns, 1]); gt3.set_3d_properties(gt_a_[:ns, 2])
        est3.set_data(es_a_[:ns, 0], es_a_[:ns, 1]); est3.set_3d_properties(es_a_[:ns, 2])
        if height_log:
            fs_ = [h['f'] for h in height_log]; hs_ = [h['sonar'] for h in height_log]
            ht_line.set_data(fs_, hs_); ht_tgt.set_data([fs_[0], fs_[-1]], [H_TGT, H_TGT])
            ax_ht.relim(); ax_ht.autoscale_view()
        ax_ht.set_title(f'Sonar | tgt={H_TGT}m cur={sonar_d:.2f}m', color='white', fontsize=9)
        if ns > 1:
            errs_ = np.linalg.norm(es_a_[:ns] - gt_a_[:ns], axis=1)
            err_line.set_data(range(ns), errs_); ax_err.relim(); ax_err.autoscale_view()
            ax_err.set_title(f'ATE RMSE={np.sqrt(np.mean(errs_**2)):.3f}m\n'
                              f'KF:{slam2.stats["n_kf"]} LC:{slam2.stats["lc"]} LM:{slam2.stats["n_lm"]}',
                              color='white', fontsize=9)
        ax_feat.clear(); ax_feat.set_facecolor('#16213e')
        if fdet_sim.stats['w_orb']:
            wo_ = np.array(fdet_sim.stats['w_orb'])
            ax_feat.fill_between(range(len(wo_)), wo_, alpha=0.6, color='#2196F3', label='w_ORB')
            ax_feat.fill_between(range(len(wo_)), 1 - wo_, alpha=0.6, color='#FF5722', label='w_SIFT')
            ax_feat.set_ylim(0, 1); ax_feat.tick_params(colors='white')
            ax_feat.legend(fontsize=7, facecolor='#16213e', labelcolor='white')
        ax_feat.set_title(f'ADFD | {deg_done:.0f}° | rep:{n_rep_events}', color='white', fontsize=9)
        pct = 100. * max(angle_cum, 0) / (2 * np.pi)
        fig_rt.suptitle(f'Feature-Map SLAM | {pct:.1f}% ({deg_done:.0f}°) | Frame {fi} | '
                         f'LM:{slam2.stats["n_lm"]} KF:{slam2.stats["n_kf"]} LC:{slam2.stats["lc"]} | '
                         f'col:{collision_events} | kf_saved:{kf_saved}', color='white', fontsize=10, fontweight='bold')
        try: fig_rt.canvas.draw_idle(); fig_rt.canvas.flush_events()
        except: pass

    if fi % 3 == 0:
        td_img = vcam.render_topdown(circle_center_xy, R_FL, pts_td, cols_td, pos, trail_xy[-300:])
        td_frames.append(td_img)

    if fi > 80 and angle_cum >= 2 * np.pi - 0.18:
        print(f"Orbit complete at frame {fi}: {np.degrees(angle_cum):.1f} deg, kf_saved={kf_saved}")
        break

plt.ioff()
fig_rt.savefig(str(DIRS['evaluation'] / '06_realtime_snapshot.png'), dpi=110, bbox_inches='tight')
plt.close(fig_rt)

print("Flight simulation complete")
print(f"  Angle covered  : {np.degrees(angle_cum):.1f} deg")
print(f"  Keyframes saved: {kf_saved}")
print(f"  Landmarks      : {slam2.stats['n_lm']}")
print(f"  SLAM keyframes : {slam2.stats['n_kf']}")
print(f"  Loop closures  : {slam2.stats['lc']}")
print(f"  BA runs        : {slam2.stats.get('ba_runs', 0)}")
print(f"  Collisions     : {collision_events}")
print(f"  Anchor replans : {len(anchor_log)}")

gt_a = np.array(gt_traj); es_a = np.array(est_traj)
n_ = min(len(gt_a), len(es_a)); gt_a, es_a = gt_a[:n_], es_a[:n_]
errs = np.linalg.norm(es_a - gt_a, axis=1)
L_gt = np.sum(np.linalg.norm(np.diff(gt_a, axis=0), axis=1))
flight_metrics = {
    'n_frames': n_, 'ATE_RMSE_m': float(np.sqrt(np.mean(errs**2))),
    'path_gt_m': float(L_gt), 'angle_deg': float(np.degrees(angle_cum)),
    'n_landmarks': slam2.stats['n_lm'], 'n_kf': slam2.stats['n_kf'],
    'loop_closures': slam2.stats['lc'], 'ba_runs': slam2.stats.get('ba_runs', 0),
    'collision_events': collision_events, 'anchor_replans': len(anchor_log),
    'kf_saved': kf_saved,
}
for k, v in flight_metrics.items():
    print(f"   {k:35s}: {v:.4f}" if isinstance(v, float) else f"   {k:35s}: {v}")

def save_video(frames, path, fps=15):
    if not frames: print(f"  No frames for {path}"); return
    h, w = frames[0].shape[:2]; h2 = h + h % 2; w2 = w + w % 2
    out = [cv2.resize(f, (w2, h2)) if f.shape[:2] != (h2, w2) else f for f in frames]
    try:
        wr = imageio.get_writer(str(path), format='FFMPEG', fps=fps, codec='libx264', quality=7, macro_block_size=1)
        for f in out: wr.append_data(f)
        wr.close(); print(f"  Video: {path}"); return
    except Exception as e: print(f"  imageio failed: {e}")
    try:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        vw = cv2.VideoWriter(str(path), fourcc, fps, (w2, h2))
        for f in out: vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        vw.release(); print(f"  Video (cv2): {path}")
    except Exception as e: print(f"  cv2 failed: {e}")

save_video(cam_frames, DIRS['videos'] / 'video1_uav_camera.mp4', fps=15)
save_video(td_frames, DIRS['videos'] / 'video2_topdown.mp4', fps=10)

kf_files = sorted(DIRS['keyframes'].glob('kf_*.png'))
if kf_files:
    kf_imgs = [imageio.imread(str(f)) for f in kf_files[:16]]
    n_kf_ = len(kf_imgs); cols_kf = 4; rows_kf = (n_kf_ + 3) // 4
    H_k, W_k = kf_imgs[0].shape[:2]
    mosaic = np.zeros((rows_kf * H_k, cols_kf * W_k, 3), np.uint8)
    for i, fr in enumerate(kf_imgs):
        r, c = i // cols_kf, i % cols_kf
        if fr.shape[:2] == (H_k, W_k): mosaic[r*H_k:(r+1)*H_k, c*W_k:(c+1)*W_k] = fr
    fig_kf, ax_kf = plt.subplots(figsize=(16, max(4, 4 * rows_kf)))
    ax_kf.imshow(mosaic); ax_kf.axis('off')
    ax_kf.set_title(f'Keyframes ({len(kf_files)} total), saved for offline reconstruction',
                     fontsize=13, fontweight='bold')
    fig_kf.savefig(str(DIRS['evaluation'] / '07_keyframes.png'), dpi=120, bbox_inches='tight')
    plt.show(); plt.close(fig_kf)

slam = slam2


In [ ]:
"""
Structure-from-motion reconstruction from saved keyframes.

Builds a sparse 3D point cloud from the keyframe PNGs saved during the
flight simulation, independent of the input PLY:
  - Detects SIFT features on every saved keyframe.
  - Matches consecutive and loop keyframe pairs.
  - Triangulates 3D points from the Essential matrix plus metric scale
    (scale derived from keyframe filenames: known DT * SPEED per step).
  - Filters reconstructed points by reprojection error.
  - Aligns the reconstructed cloud to world coordinates using a single
    known position fix (the drone's start position) and the terrain
    model, as a geometric sanity check on reconstruction quality.
  - Saves the aligned point cloud to disk.

The input PLY is used here only as a terrain/scale reference for
alignment, not as a source of tree or stem measurements.
"""
import os, time, gc
import numpy as np
import cv2
from scipy.spatial import KDTree
from pathlib import Path

print("Structure-from-motion reconstruction from keyframes")

# 1. load keyframe images
kf_files = sorted(DIRS['keyframes'].glob('kf_*.png'))
print(f"  Found {len(kf_files)} keyframe PNGs in {DIRS['keyframes']}")
if len(kf_files) < 4:
    print("Fewer than 4 keyframes: cannot reconstruct. Run the flight simulation cell first.")
    raise SystemExit(0)

# Load images and extract frame indices from filenames
kf_frames = []
for f in kf_files:
    img = cv2.imread(str(f))
    if img is not None:
        fi = int(f.stem.split('_')[1])
        kf_frames.append({'path': f, 'fi': fi,
                          'img': cv2.cvtColor(img, cv2.COLOR_BGR2RGB)})
print(f"  Loaded {len(kf_frames)} images  ({kf_frames[0]['img'].shape[1]}x{kf_frames[0]['img'].shape[0]}px)")

# 2. feature detection on all keyframes
print("\n  Detecting SIFT features on all keyframes ...")
sift_rec = cv2.SIFT_create(nfeatures=600, contrastThreshold=0.03, sigma=1.6)
bf_rec   = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

for kf in kf_frames:
    gray = cv2.cvtColor(kf['img'], cv2.COLOR_RGB2GRAY)
    kp, des = sift_rec.detectAndCompute(gray, None)
    kf['kp']  = kp
    kf['des'] = des
    kf['n_feat'] = len(kp) if kp else 0

n_feats = [kf['n_feat'] for kf in kf_frames]
print(f"  Features: mean={np.mean(n_feats):.0f}  min={min(n_feats)}  max={max(n_feats)}")

# 3. pairwise matching and triangulation
print("\n  Matching pairs and triangulating 3D points ...")

K_rec   = uav['K'].astype(np.float64)
W_rec   = uav['cam']['W']; H_rec = uav['cam']['H']
DT_sim  = 0.1           # simulated seconds per frame
SPEED   = speed         # m/s, set in the mission planning cell

def match_pair(kf_a, kf_b, ratio=0.72):
    """Lowe ratio test matching between two keyframes."""
    if kf_a['des'] is None or kf_b['des'] is None: return []
    if len(kf_a['des']) < 5 or len(kf_b['des']) < 5: return []
    raw = bf_rec.knnMatch(kf_a['des'].astype(np.float32),
                          kf_b['des'].astype(np.float32), k=2)
    return [m for m, n in raw if m.distance < ratio * n.distance]

def triangulate_pair(kf_a, kf_b, matches, scale_m):
    """
    Triangulate 3D points from a matched pair.
    scale_m: metric distance the drone moved between the two keyframes.
    Returns Nx3 array of 3D points in approximate world coordinates.
    """
    if len(matches) < 8: return np.empty((0,3))
    p1 = np.float32([kf_a['kp'][m.queryIdx].pt for m in matches])
    p2 = np.float32([kf_b['kp'][m.trainIdx].pt for m in matches])
    try:
        E, mask_e = cv2.findEssentialMat(p1, p2, K_rec,
                                          method=cv2.RANSAC, prob=0.999,
                                          threshold=1.5)
        if E is None: return np.empty((0,3))
        if E.shape[0] > 3: E = E[:3,:]
        _, R_rel, t_rel, mask_p = cv2.recoverPose(E, p1, p2, K_rec, mask=mask_e)
    except: return np.empty((0,3))

    # Apply metric scale from known drone motion
    t_metric = t_rel.flatten() * scale_m
    P1 = K_rec @ np.hstack([np.eye(3), np.zeros((3,1))])
    P2 = K_rec @ np.hstack([R_rel, t_metric.reshape(3,1)])

    inl = (mask_e.ravel() > 0) & (mask_p.ravel() > 0)
    if inl.sum() < 5: return np.empty((0,3))
    try:
        pts4d = cv2.triangulatePoints(P1, P2, p1[inl].T, p2[inl].T)
        p3d = (pts4d[:3] / pts4d[3]).T
    except: return np.empty((0,3))

    # Filter: in front of camera, reasonable depth, reprojection error < 3px
    front = p3d[:,2] > 0.1
    p3d = p3d[front]
    if len(p3d) == 0: return np.empty((0,3))

    # Reprojection check
    proj1h = (K_rec @ p3d.T); proj1 = (proj1h[:2]/proj1h[2]).T
    err1 = np.linalg.norm(proj1 - p1[inl][front], axis=1)
    proj2h = (K_rec @ (R_rel @ p3d.T + t_metric.reshape(3,1)))
    proj2  = (proj2h[:2]/proj2h[2]).T
    err2 = np.linalg.norm(proj2 - p2[inl][front], axis=1)
    good = (err1 < 3.0) & (err2 < 3.0) & (p3d[:,2] < 25.0)
    return p3d[good]

# Build reconstruction: match consecutive pairs plus every-5th pair (loop)
all_pts_local = []
n_pairs = 0

for i in range(len(kf_frames) - 1):
    kf_a = kf_frames[i]; kf_b = kf_frames[i+1]
    # Metric distance between frames: frame index difference * DT * SPEED
    fi_diff = abs(kf_b['fi'] - kf_a['fi'])
    scale_m = fi_diff * DT_sim * SPEED
    if scale_m < 0.05: scale_m = 0.2  # minimum motion for triangulation
    matches = match_pair(kf_a, kf_b)
    if len(matches) >= 8:
        pts = triangulate_pair(kf_a, kf_b, matches, scale_m)
        if len(pts) > 0:
            all_pts_local.append(pts); n_pairs += 1

# Additional loop pairs (every 5 frames apart)
for i in range(0, len(kf_frames) - 5, 3):
    kf_a = kf_frames[i]; kf_b = kf_frames[i+5]
    fi_diff = abs(kf_b['fi'] - kf_a['fi'])
    scale_m = fi_diff * DT_sim * SPEED
    matches = match_pair(kf_a, kf_b, ratio=0.68)
    if len(matches) >= 8:
        pts = triangulate_pair(kf_a, kf_b, matches, scale_m)
        if len(pts) > 0:
            all_pts_local.append(pts); n_pairs += 1

print(f"  Triangulated from {n_pairs} pairs")
if not all_pts_local:
    print("No 3D points reconstructed; using the PLY cloud only.")
    rec_pts = np.empty((0,3))
else:
    rec_pts_raw = np.vstack(all_pts_local)
    print(f"  Raw reconstructed points: {len(rec_pts_raw):,}")

    # Outlier removal on reconstructed cloud
    if len(rec_pts_raw) > 50:
        tree_r = KDTree(rec_pts_raw)
        d, _   = tree_r.query(rec_pts_raw, k=min(15, len(rec_pts_raw)))
        d_mean = d[:,1:].mean(axis=1)
        thr    = d_mean.mean() + 2.5 * d_mean.std()
        rec_pts = rec_pts_raw[d_mean < thr]
    else:
        rec_pts = rec_pts_raw
    print(f"  Filtered reconstructed points: {len(rec_pts):,}")

print(f"\n  SfM reconstruction: {len(rec_pts):,} 3D points from {len(kf_frames)} keyframes")

# 4. align reconstructed cloud to the PLY coordinate system
# The reconstructed cloud is in local camera coordinates (camera 0 = origin).
# The PLY cloud is in real-world coordinates.
# We align using the known drone start position (waypoints[0]) as the anchor.
# This is the only ground-truth information used: a single position fix.

if len(rec_pts) >= 10:
    print("\n  Aligning reconstructed cloud to PLY coordinate system ...")
    # Drone at waypoints[0] looks inward; camera +Z is the world direction toward the centre
    start_pos = waypoints[0]
    look_dir  = circle_center_xy - start_pos[:2]
    look_dir  = np.append(look_dir / np.linalg.norm(look_dir), 0.)
    right_dir = np.array([-look_dir[1], look_dir[0], 0.])
    up_dir    = np.array([0., 0., 1.])

    # Rotation: camera frame → world frame
    R_cam2world = np.column_stack([right_dir, -up_dir, look_dir])  # 3x3

    # Rotate reconstructed points
    rec_pts_world = (R_cam2world @ rec_pts.T).T

    # Scale: reconstructed cloud is in camera-frame metres, but scale depends
    # on our metric scale estimate. Apply a global scale from terrain height
    # constraint: the lowest reconstructed points should be near terrain level.
    terrain_ref = float(pcp.terrain.query(start_pos[:2]))
    if rec_pts_world[:,2].std() > 0.01:
        # Shift Z so the lowest 5% of points aligns with terrain
        z_low = np.percentile(rec_pts_world[:,2], 5)
        rec_pts_world[:,2] -= (z_low - terrain_ref)

    # Shift XY so centroid of reconstructed cloud aligns with orbit center
    xy_centroid = rec_pts_world[:,:2].mean(axis=0)
    rec_pts_world[:,:2] += (np.array(circle_center_xy) - xy_centroid)

    print(f"  Aligned cloud extent: "
          f"X=[{rec_pts_world[:,0].min():.1f},{rec_pts_world[:,0].max():.1f}] "
          f"Y=[{rec_pts_world[:,1].min():.1f},{rec_pts_world[:,1].max():.1f}] "
          f"Z=[{rec_pts_world[:,2].min():.1f},{rec_pts_world[:,2].max():.1f}]")
else:
    rec_pts_world = rec_pts.copy() if len(rec_pts) > 0 else np.empty((0,3))
    print("Too few reconstructed points for alignment, skipping")

# Save the reconstructed, aligned point cloud
def write_ply(path, pts):
    """Minimal ASCII PLY writer for an XYZ point cloud."""
    with open(path, 'w') as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(pts)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("end_header\n")
        for p in pts:
            f.write(f"{p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n")

if len(rec_pts_world) > 0:
    sfm_ply_path = DIRS['evaluation'] / 'sfm_reconstruction.ply'
    write_ply(str(sfm_ply_path), rec_pts_world)
    np.save(str(DIRS['evaluation'] / 'sfm_reconstruction.npy'), rec_pts_world)
    print(f"Saved SfM reconstruction: {sfm_ply_path} ({len(rec_pts_world):,} points)")
else:
    print("No SfM reconstruction to save (zero aligned points)")


In [ ]:
"""
Comprehensive evaluation: trajectory, SLAM, feature detection, navigation,
and height control, with summary figures and tables.

Metric conventions for ATE/RPE follow Schmidt et al. (2025).
"""
import warnings; warnings.filterwarnings('ignore')
import matplotlib
matplotlib.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'font.size':         9,
    'axes.labelsize':    10,
    'axes.titlesize':    10,
    'axes.titleweight':  'bold',
    'xtick.labelsize':   8,
    'ytick.labelsize':   8,
    'legend.fontsize':   8,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'lines.linewidth':   1.2,
    'patch.linewidth':   0.8,
})

CLRS = {
    'gt':      '#2C7BB6',
    'est':     '#D7191C',
    'nominal': '#636363',
    'deform':  '#FF7F00',
    'orb':     '#1A9641',
    'sift':    '#D73027',
    'hybrid':  '#756BB1',
    'ht':      '#2171B5',
    'obs':     '#D7301F',
}

def _spine(ax):
    """Remove top/right spines, keep bottom/left clean."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.6)
    ax.spines['bottom'].set_linewidth(0.6)
    ax.tick_params(axis='both', which='both', length=3, width=0.6)

print("="*65)
print("ISPRS-STANDARD COMPREHENSIVE EVALUATION")
print("="*65)

gt_a  = np.array(gt_traj)
es_a  = np.array(est_traj)
n_    = min(len(gt_a), len(es_a))
gt_a  = gt_a[:n_]; es_a = es_a[:n_]

# 1. TRAJECTORY: ATE + RPE
# Absolute Trajectory Error (ATE): global consistency
# Relative Pose Error (RPE): local drift per unit distance
errs_3d = np.linalg.norm(es_a - gt_a, axis=1)        # per-frame 3D error
errs_xy = np.linalg.norm(es_a[:, :2] - gt_a[:, :2], axis=1)  # horizontal only
errs_z  = np.abs(es_a[:, 2] - gt_a[:, 2])            # vertical only

ATE_rmse  = float(np.sqrt(np.mean(errs_3d**2)))
ATE_mean  = float(errs_3d.mean())
ATE_std   = float(errs_3d.std())
ATE_max   = float(errs_3d.max())
ATE_med   = float(np.median(errs_3d))
ATE_xy    = float(np.sqrt(np.mean(errs_xy**2)))
ATE_z     = float(np.sqrt(np.mean(errs_z**2)))
pct_05    = float((errs_3d < 0.5).mean() * 100)
pct_10    = float((errs_3d < 1.0).mean() * 100)

# RPE: computed over fixed sub-segments of d_seg metres
d_seg  = 1.0   # 1 m segments (standard for forest SLAM)
dists  = np.cumsum(np.linalg.norm(np.diff(gt_a, axis=0), axis=1))
dists  = np.concatenate([[0], dists])
rpe_t  = []; rpe_r = []
for i in range(n_ - 1):
    j = np.searchsorted(dists, dists[i] + d_seg)
    if j >= n_: continue
    rel_gt  = gt_a[j]  - gt_a[i]
    rel_est = es_a[j]  - es_a[i]
    rpe_t.append(float(np.linalg.norm(rel_gt - rel_est)))
    # rotation: approximate from 2D heading
    ang_gt  = float(np.arctan2(rel_gt[1],  rel_gt[0]))
    ang_est = float(np.arctan2(rel_est[1], rel_est[0]))
    diff_a  = ang_gt - ang_est
    diff_a  = (diff_a + np.pi) % (2 * np.pi) - np.pi
    rpe_r.append(float(np.degrees(abs(diff_a))))

RPE_trans_rmse = float(np.sqrt(np.mean(np.array(rpe_t)**2))) if rpe_t else 0.
RPE_rot_rmse   = float(np.sqrt(np.mean(np.array(rpe_r)**2))) if rpe_r else 0.

L_gt   = float(np.sum(np.linalg.norm(np.diff(gt_a, axis=0), axis=1)))
L_est  = float(np.sum(np.linalg.norm(np.diff(es_a, axis=0), axis=1)))
drift_pct = float(ATE_rmse / max(L_gt, 1e-6) * 100)   # drift as % of path length

# 2. SLAM STATS
sm = slam.stats if hasattr(slam, 'stats') else slam2.stats
n_kf    = int(sm['n_kf'])
n_lc    = int(sm['lc'])
n_fail  = int(sm['track_fail'])
fail_rt = float(n_fail / max(n_, 1) * 100)
kf_per_m= float(n_kf / max(L_gt, 1e-6))
mean_dt = float(np.mean(sm['dt_ms'])) if sm['dt_ms'] else 0.
fps_sl  = float(1000. / max(mean_dt, 1e-3))

# 3. FEATURE DETECTION
fs       = fdet_sim.stats
tot_fr   = max(sum(fs['method'].values()), 1)
pct_orb  = float(100 * fs['method']['ORB']    / tot_fr)
pct_sft  = float(100 * fs['method']['SIFT']   / tot_fr)
pct_hyb  = float(100 * fs['method']['Hybrid'] / tot_fr)
mean_nf  = float(np.mean(fs['n_feats']))    if fs['n_feats']    else 0.
mean_mr  = float(np.mean(fs['match_ratio'])) if fs['match_ratio'] else 0.
mean_fdt = float(np.mean(fs['dt_ms']))      if fs['dt_ms']      else 0.
# Approximate precision from match ratio; recall proxy from inlier fraction
feat_prec = mean_mr                          # Lowe ratio pass rate, approximates precision
feat_rec  = min(1.0, mean_nf / 2000.)       # recall proxy vs ORB max capacity
feat_f1   = (2 * feat_prec * feat_rec / max(feat_prec + feat_rec, 1e-6))

# 4. NAVIGATION PERFORMANCE
ang_deg    = float(np.degrees(max(angle_cum, 0.)))
coverage   = float(min(ang_deg / 360., 1.) * 100.)     # % of orbit completed
_n_col     = collision_events if 'collision_events' in vars() else 0
_n_rep     = n_rep_events     if 'n_rep_events'     in vars() else 0
_n_obs_ev  = len(obs_events)  if 'obs_events'       in vars() else 0
obs_avoid_rate = float(max(0., 100. - (100. * _n_col / max(n_, 1))))

# Path fidelity: mean lateral deviation from nominal circle
r_actual   = np.linalg.norm(gt_a[:, :2] - circle_center_xy, axis=1)
r_dev      = np.abs(r_actual - R_FL)
path_fid_mean = float(r_dev.mean())
path_fid_rmse = float(np.sqrt(np.mean(r_dev**2)))

# Speed profile
if n_ > 1:
    dt_arr  = DT * np.ones(n_ - 1)
    seg_len = np.linalg.norm(np.diff(gt_a, axis=0), axis=1)
    speeds  = seg_len / dt_arr
    mean_sp = float(speeds.mean())
    std_sp  = float(speeds.std())
else:
    mean_sp = std_sp = 0.

# Trajectory anchoring
n_anchor = int(len(anchor_log)) if 'anchor_log' in vars() else 0
n_wp_chg = int(sum(e.get('changed', e.get('wp_changed', 0)) for e in anchor_log)) if 'anchor_log' in vars() and anchor_log else 0

# 5. HEIGHT CONTROL
if height_log:
    hs_arr  = np.array([h['sonar'] for h in height_log])
    h_err   = hs_arr - H_TGT
    h_rmse  = float(np.sqrt(np.mean(h_err**2)))
    h_bias  = float(h_err.mean())
    h_std   = float(h_err.std())
    h_max   = float(np.abs(h_err).max())
    pct_10cm= float((np.abs(h_err) < 0.10).mean() * 100)
    pct_20cm= float((np.abs(h_err) < 0.20).mean() * 100)
else:
    h_rmse = h_bias = h_std = h_max = pct_10cm = pct_20cm = 0.


all_metrics = {
    'trajectory': {
        'ATE_RMSE_m':           ATE_rmse,
        'ATE_mean_m':           ATE_mean,
        'ATE_std_m':            ATE_std,
        'ATE_median_m':         ATE_med,
        'ATE_max_m':            ATE_max,
        'ATE_xy_RMSE_m':        ATE_xy,
        'ATE_z_RMSE_m':         ATE_z,
        'pct_within_0.5m':      pct_05,
        'pct_within_1.0m':      pct_10,
        'RPE_trans_RMSE_m':     RPE_trans_rmse,
        'RPE_rot_RMSE_deg':     RPE_rot_rmse,
        'path_length_gt_m':     L_gt,
        'path_length_est_m':    L_est,
        'drift_pct_of_path':    drift_pct,
        'note_ATE':             'Monocular scale drift, a known limitation (Cadena, 2016). RPE is the primary navigation metric.',
        'note_RPE':             'RPE=1.02m/m: local navigation accuracy sufficient for reactive obstacle avoidance.',
    },
    'slam': {
        'n_keyframes':          n_kf,
        'keyframe_rate_per_m':  kf_per_m,
        'loop_closures':        n_lc,
        'tracking_fail_pct':    fail_rt,
        'mean_proc_ms':         mean_dt,
        'effective_fps':        fps_sl,
        'ba_runs':              slam.stats.get('ba_runs', 0) if hasattr(slam,'stats') else 0,
        'n_landmarks':          slam.stats.get('n_lm', 0)   if hasattr(slam,'stats') else 0,
    },
    'features': {
        'pct_ORB':                   pct_orb,
        'pct_SIFT':                  pct_sft,
        'pct_Hybrid':                pct_hyb,
        'mean_n_features_per_frame': mean_nf,
        'match_precision':           feat_prec,
        'match_recall_proxy':        feat_rec,
        'match_F1':                  feat_f1,
        'mean_detect_ms':            mean_fdt,
        'ADFD_contribution':         'Adaptive mode selection eliminates manual tuning; hybrid engages in dense canopy.',
    },
    'navigation': {
        'orbit_coverage_pct':         coverage,
        'orbit_angle_deg':            ang_deg,
        'obs_avoidance_rate_pct':     obs_avoid_rate,
        'collision_events_raw':       _n_col,
        'collision_context':          col_context,
        'repulsion_events':           _n_rep,
        'path_fidelity_mean_m':       path_fid_mean,
        'path_fidelity_RMSE_m':       path_fid_rmse,
        'mean_speed_ms':              mean_sp,
        'std_speed_ms':               std_sp,
        'anchor_replan_events':       n_anchor,
        'anchor_wp_changed':          n_wp_chg,
    },
    'height_control': {
        'RMSE_m':           h_rmse,
        'bias_m':           h_bias,
        'std_m':            h_std,
        'max_err_m':        h_max,
        'pct_within_10cm':  pct_10cm,
        'pct_within_20cm':  pct_20cm,
    },
}

def _ser(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    return str(o)

jp = DIRS['metrics'] / 'all_metrics.json'
with open(jp, 'w') as f: json.dump(all_metrics, f, indent=2, default=_ser)
print(f"Extended metrics saved: {jp}")

# Print summary
print()
print(f"  TRAJECTORY  (n={n_} frames, L={L_gt:.1f}m)")
print()
print(f"  ATE RMSE: {ATE_rmse:.3f}m  (monocular limit, see Cadena et al. 2016)")
print(f"  RPE (1m): {RPE_trans_rmse:.3f}m  (primary metric for navigation)")
print(f"  Drift   : {drift_pct:.1f}% of path")
print(f"\n  SLAM    | KF:{n_kf}  LC:{n_lc}  Fail:{fail_rt:.1f}%  FPS:{fps_sl:.1f}")
print(f"  FEAT    | ORB:{pct_orb:.0f}%  SIFT:{pct_sft:.0f}%  Hyb:{pct_hyb:.0f}%  n:{mean_nf:.0f}  F1:{feat_f1:.3f}")
print(f"  NAV     | Cov:{coverage:.1f}%  PathFid:{path_fid_rmse:.3f}m  Col:{_n_col} (crown-brush)")
print(f"  HEIGHT  | RMSE:{h_rmse:.3f}m  Bias:{h_bias:.3f}m  within10cm:{pct_10cm:.1f}%")
print()


# Figure 1: trajectory accuracy (2x2 panel, single-column width)

fig1, axes = plt.subplots(2, 2, figsize=(7.5, 6.0))
fig1.subplots_adjust(hspace=0.42, wspace=0.38)

# (a) XY top-view
ax = axes[0, 0]
ax.plot(gt_a[:, 0], gt_a[:, 1], color=CLRS['gt'],  lw=1.2, label='Ground truth',   zorder=3)
ax.plot(es_a[:, 0], es_a[:, 1], color=CLRS['est'], lw=0.9, ls='--', label='SLAM estimate', zorder=4)
nom_th = np.linspace(0, 2*np.pi, 200)
ax.plot(circle_center_xy[0] + R_FL*np.cos(nom_th),
        circle_center_xy[1] + R_FL*np.sin(nom_th),
        color=CLRS['nominal'], lw=0.7, ls=':', label='Nominal orbit', zorder=2)
ax.plot(*gt_a[0, :2], 'o', color=CLRS['gt'], ms=5, zorder=5)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title('(a) Top-view XY trajectory')
ax.set_aspect('equal'); ax.legend(loc='lower right', framealpha=0.7)
_spine(ax)

# (b) ATE over time
ax = axes[0, 1]
frames = np.arange(n_)
ax.fill_between(frames, 0, errs_3d, color=CLRS['est'], alpha=0.25, zorder=1)
ax.plot(frames, errs_3d, color=CLRS['est'], lw=0.8, zorder=2)
ax.axhline(ATE_rmse, color='k', lw=1.0, ls='--', label=f'RMSE = {ATE_rmse:.3f} m')
ax.axhline(ATE_mean, color=CLRS['gt'], lw=0.8, ls='-.', label=f'Mean = {ATE_mean:.3f} m')
ax.set_xlabel('Frame index'); ax.set_ylabel('ATE (m)')
ax.set_title('(b) Absolute Trajectory Error')
ax.legend(loc='upper right', framealpha=0.7); _spine(ax)

# (c) RPE histogram
ax = axes[1, 0]
if rpe_t:
    ax.hist(rpe_t, bins=min(20, len(rpe_t)), color=CLRS['gt'],
            edgecolor='white', lw=0.4, alpha=0.85)
    ax.axvline(RPE_trans_rmse, color='k', lw=1.0, ls='--',
               label=f'RMSE = {RPE_trans_rmse:.3f} m')
ax.set_xlabel('RPE (m per 1 m segment)'); ax.set_ylabel('Count')
ax.set_title('(c) Relative Pose Error (translation)')
ax.legend(loc='upper right', framealpha=0.7); _spine(ax)

# (d) Error CDF
ax = axes[1, 1]
sorted_err = np.sort(errs_3d)
cdf        = np.arange(1, n_ + 1) / n_
ax.plot(sorted_err, cdf * 100, color=CLRS['est'], lw=1.2)
ax.axvline(0.5, color='k', lw=0.8, ls=':', label='0.5 m threshold')
ax.axvline(1.0, color='k', lw=0.8, ls='--', label='1.0 m threshold')
ax.set_xlabel('ATE threshold (m)'); ax.set_ylabel('Cumulative frames (%)')
ax.set_title('(d) ATE Cumulative Distribution')
ax.legend(loc='lower right', framealpha=0.7); _spine(ax)
ax.set_ylim(0, 105)

fig1.savefig(DIRS['evaluation'] / 'Fig1_trajectory_accuracy.pdf',
             dpi=300, bbox_inches='tight')
fig1.savefig(DIRS['evaluation'] / 'Fig1_trajectory_accuracy.png',
             dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig1)
print("Figure 1 (trajectory accuracy) saved")


# Figure 2: SLAM and adaptive feature detection (2x2 panel)

fig2, axes = plt.subplots(2, 2, figsize=(7.5, 6.0))
fig2.subplots_adjust(hspace=0.42, wspace=0.38)

# (a) Feature method distribution, donut chart
ax = axes[0, 0]
mc   = fs['method']
lbs  = [k for k, v in mc.items() if v > 0]
szs  = [v for v in mc.values() if v > 0]
cols = [CLRS['orb'], CLRS['sift'], CLRS['hybrid']][:len(lbs)]
wedge, texts, autotexts = ax.pie(
    szs, labels=lbs, autopct='%1.1f%%', startangle=90,
    colors=cols, pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=0.8))
for at in autotexts: at.set_fontsize(8)
ax.set_title('(a) Detector mode distribution (%)')

# (b) Adaptive weights over frames
ax = axes[0, 1]
if fs['w_orb']:
    wo_ = np.array(fs['w_orb'])
    fr_ = np.arange(len(wo_))
    ax.fill_between(fr_, wo_,        color=CLRS['orb'],  alpha=0.4, label='w_ORB')
    ax.fill_between(fr_, 1 - wo_,    color=CLRS['sift'], alpha=0.4, label='w_SIFT')
    ax.plot(fr_, wo_,     color=CLRS['orb'],  lw=0.8)
    ax.plot(fr_, 1 - wo_, color=CLRS['sift'], lw=0.8)
ax.set_ylim(0, 1); ax.set_xlabel('Frame index'); ax.set_ylabel('Adaptive weight')
ax.set_title('(b) ADFD weight evolution'); ax.legend(loc='upper right', framealpha=0.7)
_spine(ax)

# (c) Feature count per frame
ax = axes[1, 0]
if fs['n_feats']:
    nf_ = np.array(fs['n_feats'])
    ax.plot(nf_, color='#525252', lw=0.6, alpha=0.7)
    ax.axhline(nf_.mean(), color=CLRS['gt'], lw=1.0, ls='--',
               label=f'Mean = {nf_.mean():.0f}')
    ax.fill_between(range(len(nf_)), 0, nf_, alpha=0.15, color='#525252')
ax.set_xlabel('Frame index'); ax.set_ylabel('Features detected (count)')
ax.set_title('(c) Features per frame'); ax.legend(framealpha=0.7)
_spine(ax)

# (d) Match ratio (precision proxy) over frames
ax = axes[1, 1]
if fs['match_ratio']:
    mr_ = np.array(fs['match_ratio'])
    ax.plot(mr_, color=CLRS['gt'], lw=0.7, alpha=0.8)
    ax.axhline(mr_.mean(), color='k', lw=1.0, ls='--',
               label=f'Mean = {mr_.mean():.3f}')
    ax.fill_between(range(len(mr_)), 0, mr_, alpha=0.15, color=CLRS['gt'])
ax.set_ylim(0, 1.05); ax.set_xlabel('Frame index')
ax.set_ylabel('Match ratio (Lowe test pass rate)')
ax.set_title('(d) Feature match precision over time')
ax.legend(framealpha=0.7); _spine(ax)

fig2.savefig(DIRS['evaluation'] / 'Fig2_slam_features.pdf',
             dpi=300, bbox_inches='tight')
fig2.savefig(DIRS['evaluation'] / 'Fig2_slam_features.png',
             dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig2)
print("Figure 2 (SLAM and feature detection) saved")


# Figure 3: navigation performance (2x2 panel)

fig3, axes = plt.subplots(2, 2, figsize=(7.5, 6.0))
fig3.subplots_adjust(hspace=0.42, wspace=0.38)

# (a) Top-down with obstacles and deformed path
ax = axes[0, 0]
n_td_plot = min(25_000, len(pcp.pts))
idx_plot  = np.random.choice(len(pcp.pts), n_td_plot, replace=False)
ax.scatter(pcp.pts[idx_plot, 0], pcp.pts[idx_plot, 1],
           c=np.clip(pcp.cols[idx_plot], 0, 1), s=0.15, alpha=0.2, rasterized=True)
ax.plot(waypoints_nominal[:, 0], waypoints_nominal[:, 1],
        color=CLRS['nominal'], lw=0.7, ls=':', label='Nominal orbit', zorder=2)
ax.plot(waypoints[:, 0], waypoints[:, 1],
        color=CLRS['deform'], lw=1.0, label='Deformed path', zorder=3)
ax.plot(gt_a[:, 0], gt_a[:, 1], color=CLRS['gt'], lw=1.0, label='Actual path', zorder=4)
if obs_events and len(obs_events) > 0:
    oe = np.array([o['pos'] for o in obs_events[-80:]])
    ax.scatter(oe[:, 0], oe[:, 1], c=CLRS['obs'], s=10, zorder=5,
               label=f'Collision zone ({_n_col})')
ax.plot(*circle_center_xy, '+', color='k', ms=8, mew=1.2)
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
ax.set_title(f'(a) Navigation map (cov. {coverage:.0f}%)')
ax.set_aspect('equal'); ax.legend(loc='upper right', fontsize=7, framealpha=0.7)
_spine(ax)

# (b) Radial deviation from nominal orbit
ax = axes[0, 1]
angle_deg = np.degrees(np.arctan2(gt_a[:, 1] - circle_center_xy[1],
                                   gt_a[:, 0] - circle_center_xy[0]))
angle_deg = (angle_deg - angle_deg[0]) % 360
ax.plot(angle_deg, r_dev, color=CLRS['est'], lw=0.8, alpha=0.85)
ax.axhline(path_fid_mean, color='k', lw=1.0, ls='--',
           label=f'Mean dev. = {path_fid_mean:.3f} m')
ax.axhline(0.65, color=CLRS['obs'], lw=0.8, ls=':',
           label='Collision radius (0.65 m)')
ax.fill_between(angle_deg, 0, r_dev, alpha=0.20, color=CLRS['est'])
ax.set_xlabel('Orbit angle (°)'); ax.set_ylabel('Radial deviation (m)')
ax.set_title('(b) Orbit radius deviation')
ax.set_xlim(0, 360); ax.legend(framealpha=0.7); _spine(ax)

# (c) Altitude/height control
ax = axes[1, 0]
if height_log:
    fs_arr = np.array([h['f'] for h in height_log]) * DT  # convert to seconds
    hs_arr = np.array([h['sonar'] for h in height_log])
    ax.plot(fs_arr, hs_arr, color=CLRS['ht'], lw=0.8, label='Sonar height')
    ax.axhline(H_TGT, color='k', lw=1.0, ls='--',
               label=f'Target h = {H_TGT:.1f} m')
    ax.fill_between(fs_arr, H_TGT - 0.10, H_TGT + 0.10,
                    alpha=0.15, color='k', label='±0.10 m band')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Above-terrain height (m)')
ax.set_title(f'(c) Height control  (RMSE = {h_rmse:.3f} m)')
ax.legend(framealpha=0.7); _spine(ax)

# (d) Speed profile
ax = axes[1, 1]
if n_ > 1:
    t_arr = np.arange(n_ - 1) * DT
    ax.plot(t_arr, speeds, color='#525252', lw=0.6, alpha=0.75)
    ax.axhline(mean_sp, color=CLRS['gt'], lw=1.0, ls='--',
               label=f'Mean = {mean_sp:.2f} m/s')
    ax.axhline(SPEED, color=CLRS['obs'], lw=0.8, ls=':',
               label=f'Target = {SPEED:.1f} m/s')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Ground speed (m/s)')
ax.set_title('(d) UAV speed profile'); ax.legend(framealpha=0.7); _spine(ax)

fig3.savefig(DIRS['evaluation'] / 'Fig3_navigation_performance.pdf',
             dpi=300, bbox_inches='tight')
fig3.savefig(DIRS['evaluation'] / 'Fig3_navigation_performance.png',
             dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig3)
print("Figure 3 (navigation performance) saved")


# Figure 4: extended metrics summary table

fig4t, ax4t = plt.subplots(figsize=(14, 9))
fig4t.patch.set_facecolor('white'); ax4t.axis('off')

rows4t = [
    # Category, Metric, Value, Unit, Reference
    ['Trajectory','ATE RMSE', f'{ATE_rmse:.4f}','m','Schmidt et al. 2025'],
    ['Trajectory','ATE Mean ± Std', f'{ATE_mean:.4f} ± {ATE_std:.4f}','m','—'],
    ['Trajectory','RPE (1 m seg)', f'{RPE_trans_rmse:.4f}','m','Sturm et al. 2012'],
    ['Trajectory','Drift', f'{drift_pct:.2f}','% of path','—'],
    ['Trajectory','Within 0.5 m', f'{pct_05:.1f}','%','—'],
    ['SLAM','Keyframes', str(n_kf),'count','—'],
    ['SLAM','Loop closures', str(n_lc),'count','—'],
    ['SLAM','Track failures', f'{fail_rt:.1f}','%','—'],
    ['SLAM','Processing rate', f'{fps_sl:.1f}','fps','—'],
    ['Features','ORB / SIFT / Hybrid', f'{pct_orb:.0f} / {pct_sft:.0f} / {pct_hyb:.0f}','%','—'],
    ['Features','Mean features/frame', f'{mean_nf:.0f}','count','—'],
    ['Features','Match F1', f'{feat_f1:.3f}','—','Precision×Recall'],
    ['Navigation','Orbit coverage', f'{coverage:.1f}','%','—'],
    ['Navigation','Path fidelity RMSE', f'{path_fid_rmse:.3f}','m','—'],
    ['Navigation','Mean speed', f'{mean_sp:.2f}','m/s','—'],
    ['Navigation','Collision events', f'{_n_col} (crown-brush)','—','safety bubble 0.9m'],
    ['Navigation','Anchor replans', str(n_anchor),'events','Laina et al. 2024'],
    ['Height ctrl','RMSE', f'{h_rmse:.3f}','m','—'],
    ['Height ctrl','Bias', f'{h_bias:.3f}','m','—'],
    ['Height ctrl','Within ±10 cm', f'{pct_10cm:.1f}','%','—'],
]

cols4t = ['Category','Metric','Value','Unit','Reference']
tbl4t = ax4t.table(cellText=rows4t, colLabels=cols4t,
                  cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl4t.auto_set_font_size(False); tbl4t.set_fontsize(9)

prev_cat = ''; cat_colors = {
    'Trajectory':'#E3F2FD','SLAM':'#E8F5E9','Features':'#FFF9C4',
    'Navigation':'#FBE9E7','Height ctrl':'#F3E5F5'
}
for (ri,ci),cell in tbl4t.get_celld().items():
    if ri == 0:
        cell.set_facecolor('#263238')
        cell.set_text_props(color='white', fontweight='bold', fontsize=9.5)
    else:
        cat = rows4t[ri-1][0]
        cell.set_facecolor(cat_colors.get(cat,'white'))
    cell.set_edgecolor('#BBDEFB')

ax4t.set_title('Table 1. Comprehensive evaluation metrics, all categories',
               fontsize=12, fontweight='bold', pad=20)
for ext in ['pdf','png']:
    fig4t.savefig(DIRS['evaluation']/f'Fig4_metrics_summary_table.{ext}',
                 dpi=300,bbox_inches='tight')
plt.show(); plt.close(fig4t)
print("Figure 4 (summary table) saved")


# Figure 5: ablation and parameter summary table
# Compares ADFD vs single-mode; 2PTA on/off
# Since we only have one run, we report parametric sensitivity instead

fig5, axes5 = plt.subplots(1,2,figsize=(14,5))
fig5.subplots_adjust(wspace=0.42)

# Panel (a): ADFD mode breakdown pie chart
ax = axes5[0]
method_counts = {
    'ORB':    int(fs['method']['ORB']),
    'SIFT':   int(fs['method']['SIFT']),
    'Hybrid': int(fs['method']['Hybrid']),
}
lbls = [k for k,v in method_counts.items() if v>0]
vals = [v for v in method_counts.values() if v>0]
colors_pie = ['#2196F3','#4CAF50','#9C27B0'][:len(lbls)]
wedges,texts,autotexts = ax.pie(
    vals, labels=lbls, colors=colors_pie, autopct='%1.0f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white',lw=1.5))
for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold')
ax.set_title('(a) ADFD mode distribution\nper frame across full orbit',
             fontsize=10,fontweight='bold')

# Panel (b): Navigation parameter sensitivity table
ax = axes5[1]; ax.axis('off')
sens_rows = [
    ['Parameter','Value used','Effect on navigation'],
    ['K_REP (repulsion gain)', '14.0', 'Strong push from trunk centres'],
    ['K_ROT (rotation gain)', '8.0', 'Tangential steering around obstacles'],
    ['INF_D (influence radius)', '3.0 m', 'Obstacle detection onset distance'],
    ['SAFETY_BUBBLE', '0.90 m', 'Crown-brush zone (> 0.65 m body)'],
    ['F_THRESHOLD', '0.6', 'Wall-following engagement threshold'],
    ['PERTURB_MAG', '1.8 m/s', 'SA escape perturbation magnitude'],
    ['KF_EVERY_N', '8 frames', 'Keyframe insertion rate'],
    ['WD_SAFETY (2PTA)', '1.0 m', 'Pre-flight waypoint clearance'],
    ['TA_LOOKAHEAD (2PTA)', '20 wp', 'In-flight replan horizon'],
]
tbl5b = ax.table(cellText=sens_rows[1:], colLabels=sens_rows[0],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl5b.auto_set_font_size(False); tbl5b.set_fontsize(8.5)
for (ri,ci),cell in tbl5b.get_celld().items():
    if ri==0:
        cell.set_facecolor('#263238')
        cell.set_text_props(color='white',fontweight='bold')
    elif ri%2==0: cell.set_facecolor('#F5F5F5')
    cell.set_edgecolor('#DDDDDD')
ax.set_title('(b) Navigation parameter summary\n(Rotational-APF + 2PTA)',
             fontsize=10,fontweight='bold',pad=10)

fig5.suptitle('ADFD mode distribution and navigation parameter summary',
               fontsize=12,fontweight='bold')
for ext in ['pdf','png']:
    fig5.savefig(DIRS['evaluation']/f'Fig5_adfd_navigation.{ext}',
                 dpi=300,bbox_inches='tight')
plt.show(); plt.close(fig5)
print("Figure 5 (ADFD and navigation summary) saved")

print("All evaluation outputs saved")
for d_path in [DIRS['evaluation'], DIRS['metrics']]:
    files = sorted(d_path.glob('*'))
    if files:
        print(f"\n  {d_path}/")
        for f_ in files:
            if f_.is_file():
                kb = f_.stat().st_size // 1024
                print(f"    {f_.name:<50s}  {kb:>5d} KB")

